# Machine Learning — Customer Reactivation Propensity

## Notebook Objective

The exploratory analysis established that customer inactivity should not be treated as permanent churn.

A substantial proportion of customers returned after extended periods without purchasing, including customers who had already crossed 90-day and 180-day inactivity thresholds.

This creates a commercially important predictive problem:

> **When a previously purchasing customer reaches 90 days of inactivity, what is the probability that the customer will make another purchase within the following 90 days?**

This notebook develops a machine-learning model to estimate that probability.

The model is intended to support reactivation prioritisation.

Rather than treating all inactive customers equally, the business could use predicted reactivation propensity to distinguish between:

- customers likely to return naturally;
- customers who may benefit from targeted reactivation;
- customers with historically strong value but weakening engagement;
- and customers whose probability of return is comparatively low.

## Prediction Point

The prediction is made when a customer reaches **90 consecutive days without a qualifying purchase**.

At that moment, the model may use only information observable before or at the 90-day inactivity threshold.

No information occurring after the prediction point may be used as an input feature.

## Target

The target is:

**1 — customer makes another qualifying purchase within the next 90 days**

**0 — customer does not make another qualifying purchase within the next 90 days**

Customers whose 90-day inactivity threshold occurs too close to the end of the observed dataset to provide a complete additional 90-day outcome window are excluded from the binary modelling population.

## Key Difference from Churn Modelling

This notebook does not attempt to classify customers as permanently churned.

A customer who has not purchased for 90 days is considered **inactive**, not lost.

The model predicts the probability of subsequent reactivation.

This avoids the methodological error of defining inactivity itself as churn.

## Modelling Strategy

The notebook will:

1. load the cleaned canonical transaction dataset;
2. reconstruct identified customer purchase histories;
3. identify valid 90-day inactivity episodes;
4. define leakage-safe prediction points;
5. create the 90-day reactivation target;
6. engineer customer-history features available at the inactivity threshold;
7. split inactivity episodes chronologically into train, validation and test populations;
8. establish predictive baselines;
9. train interpretable and non-linear models;
10. compare discrimination, lift and calibration;
11. select and freeze the strongest model;
12. evaluate it once on a future temporal holdout;
13. translate reactivation scores into commercially usable customer segments.

## Business Objective

The ultimate goal is not simply to predict which inactive customers will return.

The commercial objective is to improve the allocation of reactivation effort.

The model should help answer:

> **Which inactive customers should we prioritise, when should we contact them, and where is intervention likely to generate sufficient value to justify its cost?**

## Step 31 — Reactivation Problem Architecture and Inactivity-Episode Dataset

Step 31 defines the analytical population for the customer-reactivation model.

The unit of analysis is not the customer alone.

A single customer may experience multiple periods of inactivity during their observed history.

Each eligible inactivity episode therefore represents a separate prediction opportunity.

The prediction point occurs exactly 90 days after the customer's most recent qualifying purchase.

At that point, historical behaviour up to the inactivity threshold is used to predict whether the customer purchases again during the following 90 days.

This structure is necessary because:

- inactivity can happen more than once;
- customers may reactivate after earlier inactivity episodes;
- customer behaviour may change through time;
- and the same customer can legitimately generate multiple prediction opportunities.

The modelling design must ensure that every feature is known by the prediction date and that future information does not leak into the feature set.

In [ ]:
# ================================================================
# 31.2 — IMPORT LIBRARIES
# ================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


pd.set_option(
    "display.max_columns",
    120
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.4f}"
)

In [ ]:
# ================================================================
# 31.3 — LOAD CLEANED CANONICAL DATA
# ================================================================

DATA_PATH = Path(
    "cleaned_online_retail.parquet"
)


if not DATA_PATH.exists():

    raise FileNotFoundError(
        "cleaned_online_retail.parquet was not found. "
        "Export the final canonical df from Notebook 1 first."
    )


df = pd.read_parquet(
    DATA_PATH
)


print(
    f"Rows loaded: {len(df):,}"
)

print(
    f"Columns loaded: {df.shape[1]:,}"
)

print(
    f"Observed date range: "
    f"{df['date'].min()} to {df['date'].max()}"
)

In [ ]:
# ================================================================
# 31.4 — VALIDATE REQUIRED COLUMNS
# ================================================================

required_columns = [
    "date",
    "invoice",
    "customer",
    "product",
    "region",
    "quantity",
    "unit_price",
    "customer_is_identified",
    "is_merchandise",
    "is_cancellation",
    "net_merchandise_revenue",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if missing_columns:

    raise KeyError(
        "Required columns are missing: "
        f"{missing_columns}"
    )


print(
    "✓ Required reactivation modelling fields are available."
)

In [ ]:
# ================================================================
# 31.5 — STANDARDISE DATATYPES
# ================================================================

df[
    "date"
] = pd.to_datetime(
    df[
        "date"
    ],
    errors="coerce"
)


df[
    "customer"
] = (
    df[
        "customer"
    ]
    .astype("string")
)


df[
    "invoice"
] = (
    df[
        "invoice"
    ]
    .astype("string")
)


df[
    "product"
] = (
    df[
        "product"
    ]
    .astype("string")
)


df[
    "region"
] = (
    df[
        "region"
    ]
    .astype("string")
)


assert df["date"].notna().all()


print(
    "✓ Datatypes standardised."
)

In [ ]:
# ================================================================
# 31.6 — OBSERVATION BOUNDARY
# ================================================================

modeling_end_date = pd.Timestamp(
    "2011-11-30 23:59:59"
)


modeling_data = (
    df.loc[
        df[
            "date"
        ]
        <= modeling_end_date
    ]
    .copy()
)


print(
    f"Observation end: "
    f"{modeling_end_date.date()}"
)

print(
    f"Rows retained: "
    f"{len(modeling_data):,}"
)

In [ ]:
# ================================================================
# 31.7 — QUALIFYING CUSTOMER PURCHASES
# ================================================================

positive_customer_purchases = (
    modeling_data.loc[
        modeling_data[
            "customer_is_identified"
        ]
        &
        modeling_data[
            "is_merchandise"
        ]
        &
        modeling_data[
            "quantity"
        ].gt(0)
        &
        modeling_data[
            "unit_price"
        ].gt(0)
        &
        ~modeling_data[
            "is_cancellation"
        ]
    ]
    .copy()
)


print(
    f"Qualifying purchase rows: "
    f"{len(positive_customer_purchases):,}"
)

print(
    f"Identified customers: "
    f"{positive_customer_purchases['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 31.8 — CUSTOMER ORDER HISTORY
# ================================================================

customer_orders = (
    positive_customer_purchases
    .groupby(
        [
            "customer",
            "invoice"
        ],
        as_index=False
    )
    .agg(
        order_date=(
            "date",
            "min"
        ),

        order_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        order_units=(
            "quantity",
            "sum"
        ),

        distinct_products=(
            "product",
            "nunique"
        ),

        average_unit_price=(
            "unit_price",
            "mean"
        ),

        region=(
            "region",
            "first"
        ),
    )
)


customer_orders = (
    customer_orders
    .sort_values(
        [
            "customer",
            "order_date",
            "invoice"
        ]
    )
    .reset_index(
        drop=True
    )
)


customer_orders[
    "order_number"
] = (
    customer_orders
    .groupby(
        "customer"
    )
    .cumcount()
    + 1
)


display(
    customer_orders.head()
)


print(
    f"Customer orders: "
    f"{len(customer_orders):,}"
)

In [ ]:
# ================================================================
# 31.9 — NEXT PURCHASE DATE
# ================================================================

customer_orders[
    "next_order_date"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "order_date"
    ]
    .shift(-1)
)

In [ ]:
customer_orders[
    "days_to_next_order"
] = (
    customer_orders[
        "next_order_date"
    ]
    -
    customer_orders[
        "order_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 31.10 — INACTIVITY THRESHOLD
# ================================================================

INACTIVITY_THRESHOLD_DAYS = 90

REACTIVATION_WINDOW_DAYS = 90

In [ ]:
customer_orders[
    "prediction_date"
] = (
    customer_orders[
        "order_date"
    ]
    +
    pd.to_timedelta(
        INACTIVITY_THRESHOLD_DAYS,
        unit="D"
    )
)

In [ ]:
# ================================================================
# 31.11 — IDENTIFY 90-DAY INACTIVITY EPISODES
# ================================================================

customer_orders[
    "reaches_90d_inactivity"
] = (
    customer_orders[
        "next_order_date"
    ]
    .isna()
    |
    (
        customer_orders[
            "days_to_next_order"
        ]
        >
        INACTIVITY_THRESHOLD_DAYS
    )
)

In [ ]:
# ================================================================
# 31.12 — CANDIDATE INACTIVITY EPISODES
# ================================================================

inactivity_episodes = (
    customer_orders.loc[
        customer_orders[
            "reaches_90d_inactivity"
        ]
    ]
    .copy()
)


print(
    f"Candidate 90-day inactivity episodes: "
    f"{len(inactivity_episodes):,}"
)

print(
    f"Customers represented: "
    f"{inactivity_episodes['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 31.13 — REACTIVATION FOLLOW-UP WINDOW
# ================================================================

inactivity_episodes[
    "outcome_window_end"
] = (
    inactivity_episodes[
        "prediction_date"
    ]
    +
    pd.to_timedelta(
        REACTIVATION_WINDOW_DAYS,
        unit="D"
    )
)

In [ ]:
# ================================================================
# 31.14 — COMPLETE FOLLOW-UP ELIGIBILITY
# ================================================================

inactivity_episodes[
    "complete_followup"
] = (
    inactivity_episodes[
        "outcome_window_end"
    ]
    <=
    modeling_end_date
)


print(
    f"Candidate inactivity episodes: "
    f"{len(inactivity_episodes):,}"
)

print(
    f"Complete 90-day follow-up: "
    f"{inactivity_episodes['complete_followup'].sum():,}"
)

print(
    f"Right-censored episodes excluded: "
    f"{(~inactivity_episodes['complete_followup']).sum():,}"
)

In [ ]:
eligible_inactivity_episodes = (
    inactivity_episodes.loc[
        inactivity_episodes[
            "complete_followup"
        ]
    ]
    .copy()
)

In [ ]:
# ================================================================
# 31.15 — REACTIVATION TARGET
# ================================================================

eligible_inactivity_episodes[
    "reactivates_within_90d"
] = (
    eligible_inactivity_episodes[
        "next_order_date"
    ]
    .notna()
    &
    (
        eligible_inactivity_episodes[
            "next_order_date"
        ]
        >
        eligible_inactivity_episodes[
            "prediction_date"
        ]
    )
    &
    (
        eligible_inactivity_episodes[
            "next_order_date"
        ]
        <=
        eligible_inactivity_episodes[
            "outcome_window_end"
        ]
    )
).astype(int)

In [ ]:
# ================================================================
# 31.16 — TARGET LOGIC VALIDATION
# ================================================================

positive_reactivations = (
    eligible_inactivity_episodes.loc[
        eligible_inactivity_episodes[
            "reactivates_within_90d"
        ] == 1
    ]
)


assert (
    positive_reactivations[
        "next_order_date"
    ]
    >
    positive_reactivations[
        "prediction_date"
    ]
).all()


assert (
    positive_reactivations[
        "next_order_date"
    ]
    <=
    positive_reactivations[
        "outcome_window_end"
    ]
).all()


print(
    "✓ Reactivation target logic validated."
)

In [ ]:
# ================================================================
# 31.17 — REACTIVATION TARGET BALANCE
# ================================================================

reactivation_target_summary = (
    eligible_inactivity_episodes[
        "reactivates_within_90d"
    ]
    .value_counts()
    .rename_axis(
        "reactivates_within_90d"
    )
    .reset_index(
        name="episodes"
    )
)


reactivation_target_summary[
    "share_pct"
] = (
    reactivation_target_summary[
        "episodes"
    ]
    /
    reactivation_target_summary[
        "episodes"
    ].sum()
    * 100
)


display(
    reactivation_target_summary
)

In [ ]:
# ================================================================
# 31.18 — EPISODE NUMBER
# ================================================================

eligible_inactivity_episodes = (
    eligible_inactivity_episodes
    .sort_values(
        [
            "customer",
            "prediction_date"
        ]
    )
    .reset_index(
        drop=True
    )
)


eligible_inactivity_episodes[
    "inactivity_episode_number"
] = (
    eligible_inactivity_episodes
    .groupby(
        "customer"
    )
    .cumcount()
    + 1
)

In [ ]:
display(
    eligible_inactivity_episodes[
        [
            "customer",
            "order_date",
            "prediction_date",
            "next_order_date",
            "inactivity_episode_number",
            "reactivates_within_90d",
        ]
    ]
    .head(20)
)

In [ ]:
# ================================================================
# 31.19 — EPISODE FREQUENCY
# ================================================================

episode_count_per_customer = (
    eligible_inactivity_episodes
    .groupby(
        "customer"
    )
    .size()
    .rename(
        "eligible_inactivity_episodes"
    )
)


display(
    episode_count_per_customer
    .describe()
)

In [ ]:
print(
    f"Customers with >1 eligible inactivity episode: "
    f"{(episode_count_per_customer > 1).sum():,}"
)

**Important modelling issue**

Because one customer may have multiple episodes, rows are not statistically independent in the same way as Notebook 2.
That means temporal splitting becomes even more important.
We also need to avoid accidentally placing information from a later episode into features for an earlier episode.
All features must be calculated as of each episode's prediction date.

### 31.20 — Build historical features as of prediction date

This is the heart of Notebook 3.
For every inactivity episode, we want features such as:
- historical order count;
- historical revenue;
- customer tenure;
- days since first purchase;
- average order value;
- product breadth;
- average interpurchase gap;
- maximum previous gap;
- previous inactivity episodes;
- previous reactivations;
- recent spending trajectory;
- region.
We must construct these using only history up to the order immediately preceding inactivity.

In [ ]:
# ================================================================
# 31.21 — HISTORICAL ORDER COUNT
# ================================================================

customer_orders[
    "historical_order_count"
] = (
    customer_orders[
        "order_number"
    ]
)

In [ ]:
# ================================================================
# 31.22 — CUMULATIVE HISTORICAL REVENUE
# ================================================================

customer_orders[
    "historical_revenue"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "order_revenue"
    ]
    .cumsum()
)

In [ ]:
# ================================================================
# 31.23 — HISTORICAL UNITS / PRODUCT BREADTH
# ================================================================

customer_orders[
    "historical_units"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "order_units"
    ]
    .cumsum()
)

In [ ]:
# ================================================================
# 31.24 — HISTORICAL AVERAGE ORDER VALUE
# ================================================================

customer_orders[
    "historical_average_order_value"
] = (
    customer_orders[
        "historical_revenue"
    ]
    /
    customer_orders[
        "historical_order_count"
    ]
)

In [ ]:
# ================================================================
# 31.25 — CUSTOMER TENURE
# ================================================================

customer_orders[
    "first_purchase_date"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "order_date"
    ]
    .transform(
        "min"
    )
)


customer_orders[
    "customer_tenure_days"
] = (
    customer_orders[
        "order_date"
    ]
    -
    customer_orders[
        "first_purchase_date"
    ]
).dt.days

In [ ]:
customer_orders[
    "tenure_at_prediction_days"
] = (
    customer_orders[
        "prediction_date"
    ]
    -
    customer_orders[
        "first_purchase_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 31.26 — HISTORICAL INTERPURCHASE GAPS
# ================================================================

customer_orders[
    "previous_order_date"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "order_date"
    ]
    .shift(1)
)

In [ ]:
customer_orders[
    "previous_gap_days"
] = (
    customer_orders[
        "order_date"
    ]
    -
    customer_orders[
        "previous_order_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 31.27 — HISTORICAL GAP BEHAVIOUR
# ================================================================

customer_orders[
    "historical_average_gap_days"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "previous_gap_days"
    ]
    .expanding()
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

In [ ]:
customer_orders[
    "historical_max_gap_days"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "previous_gap_days"
    ]
    .expanding()
    .max()
    .reset_index(
        level=0,
        drop=True
    )
)

In [ ]:
customer_orders[
    "historical_min_gap_days"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "previous_gap_days"
    ]
    .expanding()
    .min()
    .reset_index(
        level=0,
        drop=True
    )
)

### 31.28 — Last-order characteristics


The most recent order before inactivity may itself matter.
We already have:
- order_revenue;
- order_units;
- distinct_products;
- average_unit_price.

Rename conceptually when merged:

- last_order_value
- last_order_units
- last_order_product_count
- last_order_average_unit_price

In [ ]:
# ================================================================
# 31.29 — HISTORICAL ORDER FREQUENCY
# ================================================================

customer_orders[
    "historical_observed_tenure_days"
] = (
    customer_orders[
        "order_date"
    ]
    -
    customer_orders[
        "first_purchase_date"
    ]
).dt.days

In [ ]:
customer_orders[
    "orders_per_30_tenure_days"
] = np.where(
    customer_orders[
        "historical_observed_tenure_days"
    ] > 0,

    customer_orders[
        "historical_order_count"
    ]
    /
    customer_orders[
        "historical_observed_tenure_days"
    ]
    * 30,

    np.nan
)

In [ ]:
# ================================================================
# 31.30 — MERGE HISTORICAL FEATURES INTO EPISODES
# ================================================================

historical_order_features = (
    customer_orders[
        [
            "customer",
            "invoice",
            "historical_order_count",
            "historical_revenue",
            "historical_units",
            "historical_average_order_value",
            "first_purchase_date",
            "historical_average_gap_days",
            "historical_max_gap_days",
            "historical_min_gap_days",
            "orders_per_30_tenure_days",
            "order_revenue",
            "order_units",
            "distinct_products",
            "average_unit_price",
            "region",
        ]
    ]
    .copy()
)

In [ ]:
historical_order_features = (
    historical_order_features
    .rename(
        columns={
            "order_revenue":
                "last_order_value",

            "order_units":
                "last_order_units",

            "distinct_products":
                "last_order_product_count",

            "average_unit_price":
                "last_order_average_unit_price",

            "region":
                "last_order_region",
        }
    )
)

In [ ]:
eligible_inactivity_episodes = (
    eligible_inactivity_episodes
    .merge(
        historical_order_features,
        on=[
            "customer",
            "invoice"
        ],
        how="left"
    )
)

In [ ]:
# ================================================================
# 31.31 — TENURE AT PREDICTION
# ================================================================

eligible_inactivity_episodes[
    "tenure_at_prediction_days"
] = (
    eligible_inactivity_episodes[
        "prediction_date"
    ]
    -
    eligible_inactivity_episodes[
        "first_purchase_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 31.32 — PRIOR INACTIVITY EPISODE COUNT
# ================================================================

eligible_inactivity_episodes[
    "prior_inactivity_episode_count"
] = (
    eligible_inactivity_episodes
    .groupby(
        "customer"
    )
    .cumcount()
)

In [ ]:
# ================================================================
# 31.33 — PRIOR REACTIVATION COUNT
# ================================================================

eligible_inactivity_episodes[
    "prior_reactivation_count"
] = (
    eligible_inactivity_episodes
    .groupby(
        "customer"
    )[
        "reactivates_within_90d"
    ]
    .cumsum()
    -
    eligible_inactivity_episodes[
        "reactivates_within_90d"
    ]
)

In [ ]:
eligible_inactivity_episodes[
    "prior_reactivation_rate"
] = np.where(
    eligible_inactivity_episodes[
        "prior_inactivity_episode_count"
    ] > 0,

    eligible_inactivity_episodes[
        "prior_reactivation_count"
    ]
    /
    eligible_inactivity_episodes[
        "prior_inactivity_episode_count"
    ],

    np.nan
)

**Important caveat**

There is one subtle timing issue here.
For a prior episode's reactivation outcome to be genuinely known by the current prediction date, that prior episode's 90-day outcome window must already have completed.
Usually this will be true if the customer reactivated and generated the current history, but we should validate rather than assume.
We'll build that safeguard in the next step.

In [ ]:
# ================================================================
# 31.34 — PREDICTION-DATE CALENDAR FEATURES
# ================================================================

eligible_inactivity_episodes[
    "prediction_month"
] = (
    eligible_inactivity_episodes[
        "prediction_date"
    ]
    .dt.month
)


eligible_inactivity_episodes[
    "prediction_weekday"
] = (
    eligible_inactivity_episodes[
        "prediction_date"
    ]
    .dt.dayofweek
)


eligible_inactivity_episodes[
    "prediction_is_weekend"
] = (
    eligible_inactivity_episodes[
        "prediction_weekday"
    ]
    .isin(
        [
            5,
            6
        ]
    )
    .astype(int)
)

### 31.35 — Days from last order is intentionally constant

At the prediction point:
> days_since_last_purchase = 90

for everyone.

Therefore:
> do not use it as a feature.

It contains no discriminatory information because prediction happens at exactly the same inactivity threshold for every episode.

This is an important design difference from a generic churn model.

In [ ]:
# ================================================================
# 31.36 — INITIAL REACTIVATION FEATURE SET
# ================================================================

reactivation_numeric_features = [
    "historical_order_count",
    "historical_revenue",
    "historical_units",
    "historical_average_order_value",

    "tenure_at_prediction_days",

    "historical_average_gap_days",
    "historical_max_gap_days",
    "historical_min_gap_days",

    "orders_per_30_tenure_days",

    "last_order_value",
    "last_order_units",
    "last_order_product_count",
    "last_order_average_unit_price",

    "prior_inactivity_episode_count",
    "prior_reactivation_count",
    "prior_reactivation_rate",

    "prediction_month",
    "prediction_weekday",
    "prediction_is_weekend",
]

In [ ]:
reactivation_categorical_features = [
    "last_order_region",
]

In [ ]:
reactivation_target = (
    "reactivates_within_90d"
)

In [ ]:
# ================================================================
# 31.37 — TARGET LEAKAGE BLACKLIST
# ================================================================

reactivation_forbidden_features = [
    "next_order_date",
    "days_to_next_order",
    "outcome_window_end",
    "reactivates_within_90d",
    "future_revenue",
    "future_order_count",
    "future_product_count",
]

In [ ]:
reactivation_model_features = (
    reactivation_numeric_features
    +
    reactivation_categorical_features
)


leaked_features = (
    set(
        reactivation_model_features
    )
    &
    set(
        reactivation_forbidden_features
    )
)


assert not leaked_features, (
    f"Target leakage detected: "
    f"{leaked_features}"
)


print(
    "✓ No explicitly forbidden future variables "
    "are present in the feature set."
)

In [ ]:
# ================================================================
# 31.38 — REACTIVATION MODELLING DATASET
# ================================================================

reactivation_model_dataset = (
    eligible_inactivity_episodes[
        [
            "customer",
            "invoice",
            "prediction_date",
            "inactivity_episode_number",

            *reactivation_model_features,

            reactivation_target,
        ]
    ]
    .copy()
)


reactivation_model_dataset = (
    reactivation_model_dataset
    .sort_values(
        "prediction_date"
    )
    .reset_index(
        drop=True
    )
)


print(
    f"Eligible modelling episodes: "
    f"{len(reactivation_model_dataset):,}"
)

print(
    f"Unique customers: "
    f"{reactivation_model_dataset['customer'].nunique():,}"
)

print(
    f"Positive reactivations: "
    f"{reactivation_model_dataset[reactivation_target].sum():,}"
)

print(
    f"Reactivation rate: "
    f"{reactivation_model_dataset[reactivation_target].mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 31.39 — FEATURE MISSINGNESS
# ================================================================

reactivation_missingness = (
    reactivation_model_dataset[
        reactivation_model_features
    ]
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
    .rename(
        "missing_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
)


display(
    reactivation_missingness
)

In [ ]:
# ================================================================
# 31.40 — INACTIVITY EPISODES BY PREDICTION MONTH
# ================================================================

reactivation_model_dataset[
    "prediction_period"
] = (
    reactivation_model_dataset[
        "prediction_date"
    ]
    .dt.to_period(
        "M"
    )
)


reactivation_monthly_summary = (
    reactivation_model_dataset
    .groupby(
        "prediction_period",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        unique_customers=(
            "customer",
            "nunique"
        ),

        reactivation_rate=(
            reactivation_target,
            "mean"
        ),
    )
)


reactivation_monthly_summary[
    "reactivation_rate_pct"
] = (
    reactivation_monthly_summary[
        "reactivation_rate"
    ]
    * 100
)


display(
    reactivation_monthly_summary
)

In [ ]:
# ================================================================
# VISUAL 31.1 — INACTIVITY EPISODES OVER TIME
# ================================================================

reactivation_monthly_summary[
    "month_label"
] = (
    reactivation_monthly_summary[
        "prediction_period"
    ]
    .astype(str)
)


plt.figure(
    figsize=(13, 6)
)


sns.barplot(
    data=reactivation_monthly_summary,
    x="month_label",
    y="episodes"
)


plt.title(
    "Eligible 90-Day Inactivity Episodes by Prediction Month"
)

plt.xlabel(
    "Prediction Month"
)

plt.ylabel(
    "Inactivity Episodes"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 31.2 — REACTIVATION RATE OVER TIME
# ================================================================

plt.figure(
    figsize=(13, 6)
)


sns.lineplot(
    data=reactivation_monthly_summary,
    x="month_label",
    y="reactivation_rate_pct",
    marker="o"
)


plt.title(
    "90-Day Reactivation Rate by Prediction Month"
)

plt.xlabel(
    "Prediction Month"
)

plt.ylabel(
    "Reactivation Within Next 90 Days (%)"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 31.41 — EPISODES PER CUSTOMER
# ================================================================

episode_distribution = (
    reactivation_model_dataset
    .groupby(
        "customer"
    )
    .size()
    .value_counts()
    .sort_index()
    .rename_axis(
        "eligible_episodes_per_customer"
    )
    .reset_index(
        name="customers"
    )
)


display(
    episode_distribution
)

In [ ]:
# ================================================================
# VISUAL 31.3 — INACTIVITY EPISODES PER CUSTOMER
# ================================================================

plt.figure(
    figsize=(9, 5)
)


sns.barplot(
    data=episode_distribution,
    x="eligible_episodes_per_customer",
    y="customers"
)


plt.title(
    "Number of Eligible Inactivity Episodes per Customer"
)

plt.xlabel(
    "Eligible Inactivity Episodes"
)

plt.ylabel(
    "Customers"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 31.42 — REACTIVATION BY PRIOR ORDER COUNT
# ================================================================

reactivation_order_history = (
    reactivation_model_dataset
    .assign(
        prior_order_band=pd.cut(
            reactivation_model_dataset[
                "historical_order_count"
            ],
            bins=[
                0,
                1,
                2,
                5,
                10,
                np.inf
            ],
            labels=[
                "1",
                "2",
                "3–5",
                "6–10",
                "11+"
            ]
        )
    )
    .groupby(
        "prior_order_band",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        reactivation_rate=(
            reactivation_target,
            "mean"
        ),
    )
)


reactivation_order_history[
    "reactivation_rate_pct"
] = (
    reactivation_order_history[
        "reactivation_rate"
    ]
    * 100
)


display(
    reactivation_order_history
)

In [ ]:
# ================================================================
# VISUAL 31.4 — REACTIVATION BY ORDER HISTORY
# ================================================================

plt.figure(
    figsize=(9, 5)
)


sns.barplot(
    data=reactivation_order_history,
    x="prior_order_band",
    y="reactivation_rate_pct"
)


plt.title(
    "90-Day Reactivation Rate by Historical Order Count"
)

plt.xlabel(
    "Historical Orders at Inactivity Threshold"
)

plt.ylabel(
    "Reactivation Rate (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 31.43 — EPISODE UNIQUENESS VALIDATION
# ================================================================

duplicate_episode_keys = (
    reactivation_model_dataset
    .duplicated(
        subset=[
            "customer",
            "prediction_date"
        ]
    )
    .sum()
)


print(
    f"Duplicate customer/prediction-date episodes: "
    f"{duplicate_episode_keys:,}"
)


assert (
    duplicate_episode_keys
    == 0
)

In [ ]:
# ================================================================
# 31.44 — FOLLOW-UP VALIDATION
# ================================================================

assert (
    (
        eligible_inactivity_episodes[
            "outcome_window_end"
        ]
        <=
        modeling_end_date
    )
    .all()
)


print(
    "✓ Every modelling episode has complete "
    "90-day future follow-up."
)

In [ ]:
# ================================================================
# 31.45 — INACTIVITY THRESHOLD VALIDATION
# ================================================================

assert (
    (
        eligible_inactivity_episodes[
            "prediction_date"
        ]
        -
        eligible_inactivity_episodes[
            "order_date"
        ]
    )
    .dt.days
    .eq(
        INACTIVITY_THRESHOLD_DAYS
    )
    .all()
)


print(
    "✓ Every prediction occurs exactly "
    "90 days after the preceding purchase."
)

In [ ]:
# ================================================================
# 31.46 — STEP 31 COMPLETION CHECK
# ================================================================

assert (
    reactivation_model_dataset[
        reactivation_target
    ]
    .isin(
        [
            0,
            1
        ]
    )
    .all()
)


assert (
    len(
        reactivation_model_dataset
    )
    >
    0
)


assert (
    reactivation_model_dataset[
        "prediction_date"
    ]
    .notna()
    .all()
)


print(
    "✓ Step 31 complete."
)

print(
    "✓ Reactivation target defined."
)

print(
    "✓ Eligible inactivity episodes constructed."
)

print(
    "✓ Future-censored episodes excluded."
)

print(
    "✓ Initial historical feature set created."
)

print(
    "✓ No explicit post-prediction leakage detected."
)

### Step 31 Summary — Reactivation Dataset Architecture

Step 31 converted customer inactivity into a formal repeated-event prediction problem.

The unit of analysis is an eligible **90-day inactivity episode**, rather than a permanently churned customer.

A prediction opportunity is created when an identified customer completes 90 consecutive days without another qualifying merchandise purchase. At that point, the modelling objective is to predict whether the customer will return during the following 90 days.

### Modelling Population

The resulting dataset contains:

- **7,108 eligible inactivity episodes**
- **4,482 unique customers**
- **2,808 reactivation events**
- **4,300 non-reactivation events**
- **90-day reactivation prevalence: 39.50%**

The target is therefore reasonably balanced for binary classification.

Approximately 39.5% of eligible inactivity episodes resulted in another purchase during the following 90 days.

This reinforces the central analytical finding that **90 days of inactivity is not equivalent to permanent churn**.

A substantial proportion of inactive customers subsequently return.

### Reactivation Is a Repeated Customer State

Customers may contribute more than one inactivity episode.

Across the 4,482 customers represented in the modelling population, the average number of eligible inactivity episodes was approximately **1.59**, with some customers generating as many as **five** separate prediction opportunities.

This confirms that inactivity and reactivation can occur repeatedly within the same customer relationship.

The modelling problem should therefore be understood as predicting a customer's behaviour at a particular inactivity event rather than assigning a permanent churn label to the customer.

### Strong Relationship Between Purchase History and Reactivation

Historical customer depth already shows a substantial descriptive relationship with subsequent reactivation.

Observed 90-day reactivation rates were approximately:

- **1 historical order: 29.1%**
- **2 historical orders: 34.6%**
- **3–5 historical orders: 44.1%**
- **6–10 historical orders: 61.8%**
- **11+ historical orders: 66.5%**

Customers with deeper historical purchase relationships are therefore substantially more likely to return after reaching the 90-day inactivity threshold.

This is a predictive association rather than evidence that increasing order count would itself cause reactivation.

### Missingness Is Primarily Structural

Several historical features contain natural missingness.

Prior-reactivation rate is missing for approximately **63.1%** of episodes because many episodes occur before the customer has an earlier eligible inactivity event.

Historical interpurchase-gap measures are missing for approximately **36.9%** of episodes because customers with limited purchase histories do not yet have enough prior orders to calculate meaningful spacing statistics.

These observations should not be discarded.

The missingness itself reflects customer-history depth and will be handled within the modelling pipeline.

### Reactivation Behaviour Varies Through Time

Monthly reactivation rates vary materially across the observation period.

Rates range from approximately the mid-20% range in weaker periods to above 50% in stronger periods.

This temporal variation means that random train-test splitting would provide an unrealistic evaluation of future performance.

The reactivation model must therefore be developed and tested using chronological population splits.

### Remaining Architecture Issues

Before modelling begins, three additional safeguards are required:

1. customer histories affected by left truncation at the beginning of the dataset must be identified;
2. the final partial prediction cohort must be excluded from temporal model development;
3. prior-reactivation features must be reconstructed so that only outcomes genuinely known by the current prediction date are used.

These safeguards will be addressed before the Train / Validation / Test split is created.



## Step 32 — Leakage-Safe Feature Engineering, Episode Timing and Temporal Split

Step 31 established the inactivity-episode population and initial feature architecture.

Step 32 performs the final data-engineering safeguards required before model training.

The objectives are to:

- identify customer histories affected by left truncation;
- remove incomplete prediction-month cohorts;
- reconstruct prior-reactivation history using only information known by each prediction date;
- add richer customer-history features without future leakage;
- audit repeated customer episodes;
- and create chronological Train, Validation and Test populations.

The test population created in this step will remain untouched until the final frozen reactivation model is evaluated.

No predictive model is trained during Step 32.

In [ ]:
# ================================================================
# 32.2 — PRESERVE STEP 31 DATASET
# ================================================================

reactivation_episodes_raw = (
    eligible_inactivity_episodes
    .copy()
)


print(
    f"Starting episodes: "
    f"{len(reactivation_episodes_raw):,}"
)

print(
    f"Starting customers: "
    f"{reactivation_episodes_raw['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 32.3 — SOURCE OPENING MONTH
# ================================================================

source_start_date = (
    positive_customer_purchases[
        "date"
    ]
    .min()
)


source_opening_period = (
    source_start_date
    .to_period("M")
)


print(
    f"Source begins: {source_start_date}"
)

print(
    f"Opening source month: {source_opening_period}"
)

In [ ]:
# ================================================================
# 32.4 — LEFT-TRUNCATION FLAG
# ================================================================

customer_first_observed_purchase = (
    customer_orders
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_observed_purchase=(
            "order_date",
            "min"
        )
    )
)


customer_first_observed_purchase[
    "first_observed_period"
] = (
    customer_first_observed_purchase[
        "first_observed_purchase"
    ]
    .dt.to_period("M")
)


customer_first_observed_purchase[
    "potentially_left_truncated"
] = (
    customer_first_observed_purchase[
        "first_observed_period"
    ]
    ==
    source_opening_period
)

In [ ]:
left_truncation_summary = (
    customer_first_observed_purchase[
        "potentially_left_truncated"
    ]
    .value_counts()
    .rename_axis(
        "potentially_left_truncated"
    )
    .reset_index(
        name="customers"
    )
)


display(
    left_truncation_summary
)

In [ ]:
# ================================================================
# 32.5 — ATTACH LEFT-TRUNCATION FLAG
# ================================================================

reactivation_episodes = (
    reactivation_episodes_raw
    .merge(
        customer_first_observed_purchase[
            [
                "customer",
                "first_observed_purchase",
                "potentially_left_truncated",
            ]
        ],
        on="customer",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
assert (
    reactivation_episodes[
        "potentially_left_truncated"
    ]
    .notna()
    .all()
)

In [ ]:
# ================================================================
# 32.6 — LEFT-TRUNCATION IMPACT
# ================================================================

left_truncated_episode_summary = pd.DataFrame(
    {
        "Metric": [
            "All episodes",
            "Potentially left-truncated episodes",
            "Unaffected episodes",
        ],

        "Count": [
            len(
                reactivation_episodes
            ),

            reactivation_episodes[
                "potentially_left_truncated"
            ].sum(),

            (
                ~reactivation_episodes[
                    "potentially_left_truncated"
                ]
            ).sum(),
        ],
    }
)


display(
    left_truncated_episode_summary
)

In [ ]:
left_truncation_target_comparison = (
    reactivation_episodes
    .groupby(
        "potentially_left_truncated",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        reactivation_rate=(
            "reactivates_within_90d",
            "mean"
        ),
    )
)


left_truncation_target_comparison[
    "reactivation_rate_pct"
] = (
    left_truncation_target_comparison[
        "reactivation_rate"
    ]
    * 100
)


display(
    left_truncation_target_comparison
)

In [ ]:
# ================================================================
# 32.7 — PRIMARY NON-LEFT-TRUNCATED POPULATION
# ================================================================

reactivation_episodes = (
    reactivation_episodes.loc[
        ~reactivation_episodes[
            "potentially_left_truncated"
        ]
    ]
    .copy()
)


print(
    f"Episodes after left-truncation exclusion: "
    f"{len(reactivation_episodes):,}"
)

print(
    f"Customers retained: "
    f"{reactivation_episodes['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 32.8 — FINAL PREDICTION COHORT INSPECTION
# ================================================================

prediction_date_range_by_month = (
    reactivation_episodes
    .assign(
        prediction_period=(
            reactivation_episodes[
                "prediction_date"
            ]
            .dt.to_period("M")
        )
    )
    .groupby(
        "prediction_period",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        first_prediction_date=(
            "prediction_date",
            "min"
        ),

        last_prediction_date=(
            "prediction_date",
            "max"
        ),
    )
)


display(
    prediction_date_range_by_month.tail(6)
)

In [ ]:
# ================================================================
# 32.9 — LAST COMPLETE PREDICTION MONTH
# ================================================================

LAST_COMPLETE_PREDICTION_PERIOD = pd.Period(
    "2011-08",
    freq="M"
)


reactivation_episodes[
    "prediction_period"
] = (
    reactivation_episodes[
        "prediction_date"
    ]
    .dt.to_period("M")
)


reactivation_episodes = (
    reactivation_episodes.loc[
        reactivation_episodes[
            "prediction_period"
        ]
        <=
        LAST_COMPLETE_PREDICTION_PERIOD
    ]
    .copy()
)


print(
    f"Final modelling prediction month: "
    f"{reactivation_episodes['prediction_period'].max()}"
)

print(
    f"Episodes retained: "
    f"{len(reactivation_episodes):,}"
)

### 32.10 — Rebuild prior reactivation history safely
This is the most subtle part.

The Step 31 version used earlier episode targets. That is directionally correct, but we now need to verify that an earlier episode's outcome is actually known by the current prediction date.

For each prior episode, define the date at which its binary outcome is fully resolved.

For a positive episode, we know reactivation when the customer returns.

For a non-reactivation, we do not know the final 0 until the 90-day outcome window closes.

In [ ]:
# ================================================================
# 32.10 — OUTCOME KNOWLEDGE DATE
# ================================================================

reactivation_episodes[
    "episode_outcome_known_date"
] = np.where(
    reactivation_episodes[
        "reactivates_within_90d"
    ].eq(1),

    reactivation_episodes[
        "next_order_date"
    ],

    reactivation_episodes[
        "outcome_window_end"
    ]
)


reactivation_episodes[
    "episode_outcome_known_date"
] = pd.to_datetime(
    reactivation_episodes[
        "episode_outcome_known_date"
    ]
)

In [ ]:
# ================================================================
# 32.11 — TIMING-SAFE PRIOR REACTIVATION FEATURES
# ================================================================

reactivation_episodes = (
    reactivation_episodes
    .sort_values(
        [
            "customer",
            "prediction_date"
        ]
    )
    .reset_index(
        drop=True
    )
)


safe_prior_episode_counts = []
safe_prior_reactivation_counts = []
safe_prior_reactivation_rates = []


for row in (
    reactivation_episodes
    .itertuples()
):

    customer_history = (
        reactivation_episodes.loc[
            (
                reactivation_episodes[
                    "customer"
                ]
                ==
                row.customer
            )
            &
            (
                reactivation_episodes[
                    "prediction_date"
                ]
                <
                row.prediction_date
            )
            &
            (
                reactivation_episodes[
                    "episode_outcome_known_date"
                ]
                <=
                row.prediction_date
            )
        ]
    )


    prior_episode_count = (
        len(
            customer_history
        )
    )


    prior_reactivation_count = (
        customer_history[
            "reactivates_within_90d"
        ]
        .sum()
    )


    if prior_episode_count > 0:

        prior_reactivation_rate = (
            prior_reactivation_count
            /
            prior_episode_count
        )

    else:

        prior_reactivation_rate = np.nan


    safe_prior_episode_counts.append(
        prior_episode_count
    )

    safe_prior_reactivation_counts.append(
        prior_reactivation_count
    )

    safe_prior_reactivation_rates.append(
        prior_reactivation_rate
    )

In [ ]:
reactivation_episodes[
    "safe_prior_inactivity_episode_count"
] = (
    safe_prior_episode_counts
)


reactivation_episodes[
    "safe_prior_reactivation_count"
] = (
    safe_prior_reactivation_counts
)


reactivation_episodes[
    "safe_prior_reactivation_rate"
] = (
    safe_prior_reactivation_rates
)

In [ ]:
# ================================================================
# 32.12 — PRIOR-HISTORY LEAKAGE AUDIT
# ================================================================

prior_history_comparison = pd.DataFrame(
    {
        "Original Prior Episode Count":
            reactivation_episodes[
                "prior_inactivity_episode_count"
            ],

        "Safe Prior Episode Count":
            reactivation_episodes[
                "safe_prior_inactivity_episode_count"
            ],

        "Original Prior Reactivation Count":
            reactivation_episodes[
                "prior_reactivation_count"
            ],

        "Safe Prior Reactivation Count":
            reactivation_episodes[
                "safe_prior_reactivation_count"
            ],
    }
)


prior_history_comparison[
    "episode_count_changed"
] = (
    prior_history_comparison[
        "Original Prior Episode Count"
    ]
    !=
    prior_history_comparison[
        "Safe Prior Episode Count"
    ]
)


prior_history_comparison[
    "reactivation_count_changed"
] = (
    prior_history_comparison[
        "Original Prior Reactivation Count"
    ]
    !=
    prior_history_comparison[
        "Safe Prior Reactivation Count"
    ]
)


print(
    "Episodes where prior episode count changed: "
    f"{prior_history_comparison['episode_count_changed'].sum():,}"
)


print(
    "Episodes where prior reactivation count changed: "
    f"{prior_history_comparison['reactivation_count_changed'].sum():,}"
)

In [ ]:
# ================================================================
# 32.13 — USE ONLY TIMING-SAFE PRIOR HISTORY
# ================================================================

reactivation_episodes[
    "prior_inactivity_episode_count"
] = (
    reactivation_episodes[
        "safe_prior_inactivity_episode_count"
    ]
)


reactivation_episodes[
    "prior_reactivation_count"
] = (
    reactivation_episodes[
        "safe_prior_reactivation_count"
    ]
)


reactivation_episodes[
    "prior_reactivation_rate"
] = (
    reactivation_episodes[
        "safe_prior_reactivation_rate"
    ]
)

In [ ]:
# ================================================================
# 32.14 — HISTORY-DEPTH INDICATORS
# ================================================================

reactivation_episodes[
    "has_prior_purchase_gap"
] = (
    reactivation_episodes[
        "historical_average_gap_days"
    ]
    .notna()
    .astype(int)
)


reactivation_episodes[
    "has_prior_inactivity_episode"
] = (
    reactivation_episodes[
        "prior_inactivity_episode_count"
    ]
    .gt(0)
    .astype(int)
)

In [ ]:
# ================================================================
# 32.15 — ORDER-HISTORY TRANSFORMATIONS
# ================================================================

reactivation_episodes[
    "log1p_historical_order_count"
] = np.log1p(
    reactivation_episodes[
        "historical_order_count"
    ]
)


reactivation_episodes[
    "log1p_historical_revenue"
] = np.log1p(
    reactivation_episodes[
        "historical_revenue"
    ]
    .clip(lower=0)
)


reactivation_episodes[
    "log1p_historical_units"
] = np.log1p(
    reactivation_episodes[
        "historical_units"
    ]
    .clip(lower=0)
)

In [ ]:
# ================================================================
# 32.16 — INACTIVITY RELATIVE TO CUSTOMER'S NORMAL CADENCE
# ================================================================

reactivation_episodes[
    "inactivity_to_average_gap_ratio"
] = np.where(
    reactivation_episodes[
        "historical_average_gap_days"
    ]
    .gt(0),

    INACTIVITY_THRESHOLD_DAYS
    /
    reactivation_episodes[
        "historical_average_gap_days"
    ],

    np.nan
)

In [ ]:
# ================================================================
# 32.17 — LAST ORDER VS HISTORICAL VALUE
# ================================================================

reactivation_episodes[
    "last_to_historical_aov_ratio"
] = np.where(
    reactivation_episodes[
        "historical_average_order_value"
    ]
    .gt(0),

    reactivation_episodes[
        "last_order_value"
    ]
    /
    reactivation_episodes[
        "historical_average_order_value"
    ],

    np.nan
)

In [ ]:
# ================================================================
# 32.18 — HISTORICAL REVENUE VELOCITY
# ================================================================

reactivation_episodes[
    "revenue_per_30_tenure_days"
] = np.where(
    reactivation_episodes[
        "tenure_at_prediction_days"
    ]
    .gt(0),

    reactivation_episodes[
        "historical_revenue"
    ]
    /
    reactivation_episodes[
        "tenure_at_prediction_days"
    ]
    * 30,

    np.nan
)

In [ ]:
# ================================================================
# 32.19 — PRIOR SUCCESSFUL REACTIVATION INDICATOR
# ================================================================

reactivation_episodes[
    "has_prior_successful_reactivation"
] = (
    reactivation_episodes[
        "prior_reactivation_count"
    ]
    .gt(0)
    .astype(int)
)

In [ ]:
# ================================================================
# 32.20 — FINAL FEATURE REGISTER
# ================================================================

reactivation_numeric_features = [
    # Relationship depth
    "historical_order_count",
    "historical_revenue",
    "historical_units",
    "historical_average_order_value",
    "tenure_at_prediction_days",

    # Historical cadence
    "historical_average_gap_days",
    "historical_max_gap_days",
    "historical_min_gap_days",
    "orders_per_30_tenure_days",
    "inactivity_to_average_gap_ratio",

    # Most recent order
    "last_order_value",
    "last_order_units",
    "last_order_product_count",
    "last_order_average_unit_price",
    "last_to_historical_aov_ratio",

    # Historical economics
    "revenue_per_30_tenure_days",

    # Prior inactivity/reactivation
    "prior_inactivity_episode_count",
    "prior_reactivation_count",
    "prior_reactivation_rate",

    # History availability
    "has_prior_purchase_gap",
    "has_prior_inactivity_episode",
    "has_prior_successful_reactivation",

    # Nonlinear depth alternatives
    "log1p_historical_order_count",
    "log1p_historical_revenue",
    "log1p_historical_units",

    # Calendar
    "prediction_month",
    "prediction_weekday",
    "prediction_is_weekend",
]


reactivation_categorical_features = [
    "last_order_region",
]


reactivation_model_features = (
    reactivation_numeric_features
    +
    reactivation_categorical_features
)

In [ ]:
# ================================================================
# 32.21 — REBUILD LEAKAGE-SAFE MODELLING TABLE
# ================================================================

reactivation_model_dataset = (
    reactivation_episodes[
        [
            "customer",
            "invoice",
            "prediction_date",
            "prediction_period",
            "inactivity_episode_number",

            *reactivation_model_features,

            reactivation_target,
        ]
    ]
    .copy()
)


reactivation_model_dataset = (
    reactivation_model_dataset
    .sort_values(
        [
            "prediction_date",
            "customer"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Final Step 32 episodes: "
    f"{len(reactivation_model_dataset):,}"
)

print(
    f"Unique customers: "
    f"{reactivation_model_dataset['customer'].nunique():,}"
)

print(
    f"Reactivation prevalence: "
    f"{reactivation_model_dataset[reactivation_target].mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 32.22 — MONTHLY TARGET STABILITY
# ================================================================

monthly_reactivation_after_filters = (
    reactivation_model_dataset
    .groupby(
        "prediction_period",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        customers=(
            "customer",
            "nunique"
        ),

        reactivation_rate=(
            reactivation_target,
            "mean"
        ),
    )
)


monthly_reactivation_after_filters[
    "reactivation_rate_pct"
] = (
    monthly_reactivation_after_filters[
        "reactivation_rate"
    ]
    * 100
)


display(
    monthly_reactivation_after_filters
)

In [ ]:
# ================================================================
# 32.23 — REPEATED CUSTOMER EPISODES
# ================================================================

episodes_per_customer_final = (
    reactivation_model_dataset
    .groupby(
        "customer"
    )
    .size()
)


repeated_customer_summary = pd.DataFrame(
    {
        "Metric": [
            "Unique customers",
            "Customers with one episode",
            "Customers with multiple episodes",
            "Maximum episodes per customer",
        ],

        "Value": [
            episodes_per_customer_final.size,

            (
                episodes_per_customer_final
                .eq(1)
                .sum()
            ),

            (
                episodes_per_customer_final
                .gt(1)
                .sum()
            ),

            episodes_per_customer_final.max(),
        ],
    }
)


display(
    repeated_customer_summary
)

### 32.24 — Temporal split philosophy

For Notebook 2 we split by acquisition month.
For Notebook 3 we split by prediction month.
This is the correct temporal unit because the score is generated when the inactivity threshold is reached.
Do not randomly split episodes.

In [ ]:
# ================================================================
# 32.25 — TEMPORAL SPLIT SUPPORT TABLE
# ================================================================

prediction_month_counts = (
    reactivation_model_dataset
    .groupby(
        "prediction_period",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "episodes"
        }
    )
)


prediction_month_counts[
    "cumulative_episodes"
] = (
    prediction_month_counts[
        "episodes"
    ]
    .cumsum()
)


prediction_month_counts[
    "cumulative_share"
] = (
    prediction_month_counts[
        "cumulative_episodes"
    ]
    /
    prediction_month_counts[
        "episodes"
    ]
    .sum()
)


display(
    prediction_month_counts
)

In [ ]:
# ================================================================
# 32.26 — TEMPORAL SPLIT BOUNDARIES
# ================================================================

train_cutoff_row = (
    prediction_month_counts.loc[
        prediction_month_counts[
            "cumulative_share"
        ]
        >= 0.70
    ]
    .iloc[0]
)


validation_cutoff_row = (
    prediction_month_counts.loc[
        prediction_month_counts[
            "cumulative_share"
        ]
        >= 0.85
    ]
    .iloc[0]
)


TRAIN_END_PERIOD = (
    train_cutoff_row[
        "prediction_period"
    ]
)


VALIDATION_END_PERIOD = (
    validation_cutoff_row[
        "prediction_period"
    ]
)


print(
    f"Training ends: {TRAIN_END_PERIOD}"
)

print(
    f"Validation ends: {VALIDATION_END_PERIOD}"
)

print(
    f"Test begins after: {VALIDATION_END_PERIOD}"
)

In [ ]:
# ================================================================
# 32.27 — TEMPORAL DATA SPLIT
# ================================================================

train_reactivation_data = (
    reactivation_model_dataset.loc[
        reactivation_model_dataset[
            "prediction_period"
        ]
        <=
        TRAIN_END_PERIOD
    ]
    .copy()
)


validation_reactivation_data = (
    reactivation_model_dataset.loc[
        (
            reactivation_model_dataset[
                "prediction_period"
            ]
            >
            TRAIN_END_PERIOD
        )
        &
        (
            reactivation_model_dataset[
                "prediction_period"
            ]
            <=
            VALIDATION_END_PERIOD
        )
    ]
    .copy()
)


test_reactivation_data = (
    reactivation_model_dataset.loc[
        reactivation_model_dataset[
            "prediction_period"
        ]
        >
        VALIDATION_END_PERIOD
    ]
    .copy()
)

In [ ]:
# ================================================================
# 32.28 — SPLIT SUMMARY
# ================================================================

reactivation_split_summary = pd.DataFrame(
    [
        {
            "Dataset":
                "Train",

            "Episodes":
                len(train_reactivation_data),

            "Unique Customers":
                train_reactivation_data[
                    "customer"
                ].nunique(),

            "Start Period":
                train_reactivation_data[
                    "prediction_period"
                ].min(),

            "End Period":
                train_reactivation_data[
                    "prediction_period"
                ].max(),

            "Reactivation Rate":
                train_reactivation_data[
                    reactivation_target
                ].mean(),
        },

        {
            "Dataset":
                "Validation",

            "Episodes":
                len(validation_reactivation_data),

            "Unique Customers":
                validation_reactivation_data[
                    "customer"
                ].nunique(),

            "Start Period":
                validation_reactivation_data[
                    "prediction_period"
                ].min(),

            "End Period":
                validation_reactivation_data[
                    "prediction_period"
                ].max(),

            "Reactivation Rate":
                validation_reactivation_data[
                    reactivation_target
                ].mean(),
        },

        {
            "Dataset":
                "Test",

            "Episodes":
                len(test_reactivation_data),

            "Unique Customers":
                test_reactivation_data[
                    "customer"
                ].nunique(),

            "Start Period":
                test_reactivation_data[
                    "prediction_period"
                ].min(),

            "End Period":
                test_reactivation_data[
                    "prediction_period"
                ].max(),

            "Reactivation Rate":
                test_reactivation_data[
                    reactivation_target
                ].mean(),
        },
    ]
)


reactivation_split_summary[
    "Reactivation Rate (%)"
] = (
    reactivation_split_summary[
        "Reactivation Rate"
    ]
    * 100
)


display(
    reactivation_split_summary
)

In [ ]:
# ================================================================
# 32.29 — TEMPORAL SEPARATION VALIDATION
# ================================================================

assert (
    train_reactivation_data[
        "prediction_date"
    ]
    .max()
    <
    validation_reactivation_data[
        "prediction_date"
    ]
    .min()
)


assert (
    validation_reactivation_data[
        "prediction_date"
    ]
    .max()
    <
    test_reactivation_data[
        "prediction_date"
    ]
    .min()
)


print(
    "✓ Strict chronological separation confirmed."
)

In [ ]:
# ================================================================
# 32.30 — CUSTOMER OVERLAP ACROSS TIME SPLITS
# ================================================================

train_customers = set(
    train_reactivation_data[
        "customer"
    ]
)


validation_customers = set(
    validation_reactivation_data[
        "customer"
    ]
)


test_customers = set(
    test_reactivation_data[
        "customer"
    ]
)


customer_overlap_summary = pd.DataFrame(
    {
        "Overlap": [
            "Train ∩ Validation",
            "Train ∩ Test",
            "Validation ∩ Test",
        ],

        "Customers": [
            len(
                train_customers
                &
                validation_customers
            ),

            len(
                train_customers
                &
                test_customers
            ),

            len(
                validation_customers
                &
                test_customers
            ),
        ],
    }
)


display(
    customer_overlap_summary
)

In [ ]:
# ================================================================
# 32.31 — TRAIN / VALIDATION / TEST MATRICES
# ================================================================

X_reactivation_train = (
    train_reactivation_data[
        reactivation_model_features
    ]
    .copy()
)


y_reactivation_train = (
    train_reactivation_data[
        reactivation_target
    ]
    .copy()
)


X_reactivation_validation = (
    validation_reactivation_data[
        reactivation_model_features
    ]
    .copy()
)


y_reactivation_validation = (
    validation_reactivation_data[
        reactivation_target
    ]
    .copy()
)


X_reactivation_test = (
    test_reactivation_data[
        reactivation_model_features
    ]
    .copy()
)


y_reactivation_test = (
    test_reactivation_data[
        reactivation_target
    ]
    .copy()
)

In [ ]:
# ================================================================
# 32.32 — TEST-SET GOVERNANCE
# ================================================================

REACTIVATION_TEST_SET_OPENED = False


print(
    "Reactivation test set created."
)

print(
    "Test status: UNOPENED"
)

print(
    "Do not evaluate any model on "
    "X_reactivation_test until the final model is frozen."
)

In [ ]:
# ================================================================
# 32.33 — FINAL FEATURE LEAKAGE AUDIT
# ================================================================

forbidden_terms = [
    "next_order",
    "outcome_window",
    "future_",
    "reactivates_within_90d",
]


suspicious_features = [
    feature
    for feature in reactivation_model_features
    if any(
        term in feature
        for term in forbidden_terms
    )
]


assert not suspicious_features, (
    "Potential leakage features found: "
    f"{suspicious_features}"
)


print(
    "✓ No obvious future-target variables in final feature set."
)

In [ ]:
# ================================================================
# 32.34 — TRAIN / VALIDATION MISSINGNESS
# ================================================================

train_missingness = (
    X_reactivation_train
    .isna()
    .mean()
    .mul(100)
    .rename(
        "train_missing_pct"
    )
)


validation_missingness = (
    X_reactivation_validation
    .isna()
    .mean()
    .mul(100)
    .rename(
        "validation_missing_pct"
    )
)


reactivation_split_missingness = (
    pd.concat(
        [
            train_missingness,
            validation_missingness,
        ],
        axis=1
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
)


display(
    reactivation_split_missingness
    .sort_values(
        "train_missing_pct",
        ascending=False
    )
)

In [ ]:
# ================================================================
# 32.35 — TRAIN → VALIDATION FEATURE DRIFT
# ================================================================

reactivation_drift_records = []


for feature in reactivation_numeric_features:

    train_values = pd.to_numeric(
        X_reactivation_train[
            feature
        ],
        errors="coerce"
    )


    validation_values = pd.to_numeric(
        X_reactivation_validation[
            feature
        ],
        errors="coerce"
    )


    train_mean = (
        train_values.mean()
    )


    validation_mean = (
        validation_values.mean()
    )


    pooled_std = np.sqrt(
        (
            train_values.std() ** 2
            +
            validation_values.std() ** 2
        )
        / 2
    )


    if (
        pd.isna(pooled_std)
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            validation_mean
            -
            train_mean
        ) / pooled_std


    reactivation_drift_records.append(
        {
            "feature":
                feature,

            "train_mean":
                train_mean,

            "validation_mean":
                validation_mean,

            "SMD":
                smd,

            "absolute_SMD":
                abs(smd)
                if not pd.isna(smd)
                else np.nan,
        }
    )


reactivation_feature_drift = (
    pd.DataFrame(
        reactivation_drift_records
    )
    .sort_values(
        "absolute_SMD",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    reactivation_feature_drift
)

In [ ]:
plt.figure(
    figsize=(11, 9)
)


sns.barplot(
    data=(
        reactivation_feature_drift
        .sort_values(
            "absolute_SMD"
        )
    ),
    y="feature",
    x="absolute_SMD"
)


plt.axvline(
    0.10,
    linestyle="--",
    label="0.10"
)


plt.axvline(
    0.25,
    linestyle="--",
    label="0.25"
)


plt.title(
    "Reactivation Model — Train-to-Validation Feature Drift"
)

plt.xlabel(
    "Absolute Standardised Mean Difference"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 32.36 — STEP 32 COMPLETION CHECK
# ================================================================

assert (
    len(
        train_reactivation_data
    )
    > 0
)


assert (
    len(
        validation_reactivation_data
    )
    > 0
)


assert (
    len(
        test_reactivation_data
    )
    > 0
)


assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


assert (
    reactivation_model_dataset[
        reactivation_target
    ]
    .isin(
        [0, 1]
    )
    .all()
)


print(
    "✓ Step 32 complete."
)

print(
    "✓ Left-truncated histories excluded from primary model."
)

print(
    "✓ Partial final prediction cohort excluded."
)

print(
    "✓ Prior reactivation history reconstructed using timing-safe information."
)

print(
    "✓ Final reactivation feature architecture created."
)

print(
    "✓ Chronological Train / Validation / Test split created."
)

print(
    "✓ Reactivation Test set remains unopened."
)

### Step 32 Summary — Leakage-Safe Feature Architecture and Temporal Split

Step 32 completed the final population and feature-engineering safeguards required before training the reactivation model.

### Left-Truncated Customer Histories Were Material

Of the original 7,108 eligible inactivity episodes, **1,642 episodes** were associated with customers first observed during the opening month of the source dataset.

These histories may be incomplete because transactions occurring before the dataset begins cannot be observed.

This is particularly important for the reactivation model because historical relationship depth — including order count, revenue, tenure and purchase spacing — is expected to contain substantial predictive information.

Potentially left-truncated customers were therefore excluded from the primary modelling population.

The primary analysis retains only customers whose observable relationship begins after the source opening month.

### Timing-Safe Prior Reactivation Features Were Confirmed

Prior inactivity and reactivation variables were reconstructed using the date on which each earlier episode's outcome would actually have become known.

The timing-safe reconstruction changed:

- **0 prior episode counts**
- **0 prior reactivation counts**

This confirms that the initial prior-reactivation logic was already temporally valid for the retained dataset.

No correction to historical target information was required.

### The Partial Final Prediction Cohort Was Removed

The September 2011 prediction cohort contained only a small number of episodes because only the earliest September inactivity thresholds had sufficient future follow-up before the observation boundary.

September 2011 was therefore excluded from the primary model.

The final complete prediction month is **August 2011**.

### Final Modelling Population

After left-truncation and prediction-cohort safeguards, the temporal populations contain:

- **Train: 4,052 inactivity episodes**
- **Validation: 802 inactivity episodes**
- **Test: 599 inactivity episodes**

The corresponding 90-day reactivation rates are approximately:

- **Train: 34.58%**
- **Validation: 37.03%**
- **Test: 43.74%**

The increasing future reactivation prevalence demonstrates that the outcome distribution itself changes through time.

For this reason, future PR-AUC results must be interpreted alongside prevalence, ROC-AUC and commercial lift.

### Customer Reactivation Is a Repeated Event

The final modelling population represents **3,672 unique customers**.

Of these:

- **2,284 customers have one eligible inactivity episode**
- **1,388 customers have multiple eligible episodes**
- the maximum is **five episodes for one customer**

This confirms that customer inactivity and reactivation are recurring behavioural states rather than one-time churn outcomes.

### Customer-History Maturity Changes Materially Through Time

Train-to-validation drift is substantial for several accumulated relationship variables.

The largest changes include:

- tenure at prediction — **absolute SMD 0.86**
- prior inactivity episode count — **0.72**
- historical maximum purchase gap — **0.72**
- prior-inactivity indicator — **0.62**
- log historical order count — **0.55**
- historical order count — **0.51**
- prior successful-reactivation indicator — **0.48**

These changes are economically and temporally coherent.

As calendar time progresses, customers naturally have more opportunity to accumulate orders, tenure, inactivity episodes and historical purchase gaps.

The drift is therefore not automatically evidence of data failure.

It represents a genuine out-of-time generalisation challenge.

### Relative Behavioural Features Are More Stable

Several customer-normalised variables display considerably lower drift.

Examples include:

- last-order value relative to historical average order value;
- revenue per 30 tenure days;
- last-order product count;
- last-order units;
- and historical average order value.

This suggests that relative behavioural characteristics may generalise more consistently through time than raw accumulated customer-history measures.

The modelling stage should therefore compare a full-history specification against a more temporally stable behavioural specification.

### Final Modelling Design

The reactivation model will be developed chronologically.

The Train population contains the earliest prediction episodes, Validation contains later episodes, and the Test population contains the latest complete prediction cohorts.

The Test population remains unopened.

The next stage will establish predictive baselines and determine whether customer-history information can materially outperform simple reactivation-rate benchmarks.



## Step 33 — Baselines, Preprocessing and First Reactivation Logistic Models

Step 33 begins predictive modelling of 90-day customer reactivation.

The purpose is to establish whether customer history available at the 90-day inactivity threshold provides meaningful predictive information beyond simple baseline assumptions.

Four specifications will be evaluated initially:

1. **Training Prevalence Baseline** — every episode receives the historical training reactivation rate;
2. **Historical Order Count Baseline** — a single-feature Logistic Regression using relationship depth;
3. **Full-History Logistic Regression** — uses the complete leakage-safe feature architecture;
4. **Stable-Behaviour Logistic Regression** — emphasises behavioural rates and relative measures that exhibited lower temporal drift.

All models are trained on the historical Train population and evaluated on the later Validation population.

The final Test population remains untouched.

### Primary Evaluation Metric

PR-AUC remains the primary model-selection metric because the task is binary and future reactivation prevalence varies through time.

However, PR-AUC will always be interpreted relative to the validation prevalence baseline.

Additional metrics include:

- ROC-AUC;
- Brier score;
- precision;
- recall;
- F1;
- Lift@10%;
- Lift@20%;
- Capture@10%;
- Capture@20%;
- and score-decile separation.

The objective is not simply to maximise classification accuracy.

The commercial objective is to rank inactive customers so that reactivation effort can be prioritised more intelligently.

In [ ]:
# ================================================================
# 33.2 — MODELLING IMPORTS
# ================================================================

from sklearn.base import clone

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    roc_curve,
)

import numpy as np
import pandas as pd

In [ ]:
# ================================================================
# 33.3 — TEST-SET GOVERNANCE
# ================================================================

assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Reactivation Test population remains unopened."
)

print(
    "Step 33 will use Train + Validation only."
)

In [ ]:
# ================================================================
# 33.4 — TRAIN / VALIDATION TARGET PREVALENCE
# ================================================================

reactivation_prevalence_summary = pd.DataFrame(
    {
        "Dataset": [
            "Train",
            "Validation",
        ],

        "Episodes": [
            len(
                y_reactivation_train
            ),

            len(
                y_reactivation_validation
            ),
        ],

        "Reactivation Rate": [
            y_reactivation_train.mean(),
            y_reactivation_validation.mean(),
        ],
    }
)


reactivation_prevalence_summary[
    "Reactivation Rate (%)"
] = (
    reactivation_prevalence_summary[
        "Reactivation Rate"
    ]
    * 100
)


display(
    reactivation_prevalence_summary
)

In [ ]:
# ================================================================
# 33.5 — TIE-SAFE BASELINE LIFT / CAPTURE
# ================================================================

def lift_at_fraction(
    y_true,
    y_probability,
    fraction=0.10
):

    y_true = np.asarray(
        y_true
    )

    y_probability = np.asarray(
        y_probability
    )


    # ------------------------------------------------------------
    # Constant-score model has no ranking ability.
    # ------------------------------------------------------------

    if np.allclose(
        y_probability,
        y_probability[0]
    ):

        return 1.0


    evaluation = pd.DataFrame(
        {
            "actual":
                y_true,

            "probability":
                y_probability,
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(
            drop=True
        )
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    overall_rate = (
        evaluation[
            "actual"
        ]
        .mean()
    )


    top_rate = (
        evaluation
        .head(
            n_top
        )[
            "actual"
        ]
        .mean()
    )


    if overall_rate == 0:

        return np.nan


    return (
        top_rate
        /
        overall_rate
    )

In [ ]:
def capture_at_fraction(
    y_true,
    y_probability,
    fraction=0.10
):

    y_true = np.asarray(
        y_true
    )

    y_probability = np.asarray(
        y_probability
    )


    if np.allclose(
        y_probability,
        y_probability[0]
    ):

        return fraction


    evaluation = pd.DataFrame(
        {
            "actual":
                y_true,

            "probability":
                y_probability,
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(
            drop=True
        )
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(evaluation)
                *
                fraction
            )
        )
    )


    total_positives = (
        evaluation[
            "actual"
        ]
        .sum()
    )


    if total_positives == 0:

        return np.nan


    top_positives = (
        evaluation
        .head(
            n_top
        )[
            "actual"
        ]
        .sum()
    )


    return (
        top_positives
        /
        total_positives
    )

In [ ]:
# ================================================================
# 33.6 — REUSABLE BINARY MODEL EVALUATION
# ================================================================

def evaluate_reactivation_model(
    model_name,
    y_true,
    probabilities,
    threshold=0.50
):

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )


    predictions = (
        probabilities
        >=
        threshold
    ).astype(int)


    results = pd.DataFrame(
        {
            "Model": [
                model_name
            ],

            "PR-AUC": [
                average_precision_score(
                    y_true,
                    probabilities
                )
            ],

            "ROC-AUC": [
                roc_auc_score(
                    y_true,
                    probabilities
                )
            ],

            "Brier Score": [
                brier_score_loss(
                    y_true,
                    probabilities
                )
            ],

            "Precision": [
                precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "Recall": [
                recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "F1": [
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ],

            "Lift @ 10%": [
                lift_at_fraction(
                    y_true,
                    probabilities,
                    fraction=0.10
                )
            ],

            "Lift @ 20%": [
                lift_at_fraction(
                    y_true,
                    probabilities,
                    fraction=0.20
                )
            ],

            "Capture @ 10%": [
                capture_at_fraction(
                    y_true,
                    probabilities,
                    fraction=0.10
                )
            ],

            "Capture @ 20%": [
                capture_at_fraction(
                    y_true,
                    probabilities,
                    fraction=0.20
                )
            ],
        }
    )


    return results

In [ ]:
# ================================================================
# 33.7 — SCORE-DECILE PERFORMANCE
# ================================================================

def create_reactivation_decile_performance(
    y_true,
    probabilities,
    n_deciles=10
):

    evaluation = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    y_true
                ),

            "probability":
                np.asarray(
                    probabilities
                ),
        }
    )


    evaluation = (
        evaluation
        .sort_values(
            "probability",
            ascending=False,
            kind="mergesort"
        )
        .reset_index(
            drop=True
        )
    )


    evaluation[
        "score_decile"
    ] = (
        pd.qcut(
            np.arange(
                len(evaluation)
            ),
            q=n_deciles,
            labels=False
        )
        + 1
    )


    overall_rate = (
        evaluation[
            "actual"
        ]
        .mean()
    )


    decile_table = (
        evaluation
        .groupby(
            "score_decile",
            as_index=False
        )
        .agg(
            episodes=(
                "actual",
                "size"
            ),

            actual_reactivation_rate=(
                "actual",
                "mean"
            ),

            average_predicted_probability=(
                "probability",
                "mean"
            ),

            reactivations=(
                "actual",
                "sum"
            ),
        )
    )


    decile_table[
        "actual_reactivation_rate_pct"
    ] = (
        decile_table[
            "actual_reactivation_rate"
        ]
        * 100
    )


    decile_table[
        "average_predicted_probability_pct"
    ] = (
        decile_table[
            "average_predicted_probability"
        ]
        * 100
    )


    decile_table[
        "lift"
    ] = (
        decile_table[
            "actual_reactivation_rate"
        ]
        /
        overall_rate
    )


    return decile_table

In [ ]:
# ================================================================
# 33.8 — FULL FEATURE PREPROCESSOR
# ================================================================

reactivation_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True # because missingness itself contains information about history depth.
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),
    ]
)

In [ ]:
reactivation_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        ),
    ]
)

In [ ]:
reactivation_full_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            reactivation_numeric_transformer,
            reactivation_numeric_features,
        ),

        (
            "categorical",
            reactivation_categorical_transformer,
            reactivation_categorical_features,
        ),
    ]
)

### 33.9 — Baseline 1: Training prevalence

In [ ]:
# ================================================================
# 33.9 — BASELINE 1: TRAINING PREVALENCE
# ================================================================

training_reactivation_prevalence = (
    y_reactivation_train.mean()
)


validation_prevalence_probabilities = np.full(
    shape=len(
        y_reactivation_validation
    ),

    fill_value=training_reactivation_prevalence
)


prevalence_baseline_results = (
    evaluate_reactivation_model(
        model_name="Training Prevalence Baseline",
        y_true=y_reactivation_validation,
        probabilities=validation_prevalence_probabilities,
    )
)


display(
    prevalence_baseline_results
)

### 33.10 — Baseline 2: Historical order count only

In [ ]:
# ================================================================
# 33.10 — SINGLE-FEATURE ORDER-DEPTH BASELINE
# ================================================================

single_feature_reactivation_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median"
                            )
                        ),

                        (
                            "scaler",
                            StandardScaler()
                        ),
                    ]
                ),

                [
                    "historical_order_count"
                ],
            )
        ]
    )
)

In [ ]:
single_feature_reactivation_model = Pipeline(
    steps=[
        (
            "preprocessing",
            single_feature_reactivation_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
single_feature_reactivation_model.fit(
    X_reactivation_train[
        [
            "historical_order_count"
        ]
    ],
    y_reactivation_train
)

In [ ]:
validation_single_feature_probabilities = (
    single_feature_reactivation_model
    .predict_proba(
        X_reactivation_validation[
            [
                "historical_order_count"
            ]
        ]
    )[:, 1]
)

In [ ]:
single_feature_reactivation_results = (
    evaluate_reactivation_model(
        model_name="Historical Order Count Baseline",
        y_true=y_reactivation_validation,
        probabilities=validation_single_feature_probabilities,
    )
)


display(
    single_feature_reactivation_results
)

### 33.11 — Full-history Logistic Regression

In [ ]:
# ================================================================
# 33.11 — FULL-HISTORY LOGISTIC REGRESSION
# ================================================================

full_reactivation_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            reactivation_full_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)

In [ ]:
full_reactivation_logistic_model.fit(
    X_reactivation_train,
    y_reactivation_train
)

In [ ]:
validation_full_reactivation_probabilities = (
    full_reactivation_logistic_model
    .predict_proba(
        X_reactivation_validation
    )[:, 1]
)

In [ ]:
full_reactivation_logistic_results = (
    evaluate_reactivation_model(
        model_name="Full-History Logistic Regression",
        y_true=y_reactivation_validation,
        probabilities=validation_full_reactivation_probabilities,
    )
)


display(
    full_reactivation_logistic_results
)

### 33.12 — Define a temporally stable challenger feature set

Because Step 32 found very strong drift in cumulative customer-depth variables, create a challenger relying more heavily on rates, ratios and recent behaviour.

This is not because those features are necessarily better.

It is an explicit generalisation test.

In [ ]:
# ================================================================
# 33.12 — STABLE-BEHAVIOUR FEATURE SET
# ================================================================

stable_reactivation_numeric_features = [
    # Recent order characteristics
    "last_order_value",
    "last_order_units",
    "last_order_product_count",
    "last_order_average_unit_price",

    # Relative economics
    "historical_average_order_value",
    "last_to_historical_aov_ratio",
    "revenue_per_30_tenure_days",

    # Cadence-normalised behaviour
    "orders_per_30_tenure_days",
    "inactivity_to_average_gap_ratio",
    "prior_reactivation_rate",

    # History-availability information
    "has_prior_purchase_gap",
    "has_prior_inactivity_episode",
    "has_prior_successful_reactivation",

    # Calendar
    "prediction_month",
    "prediction_weekday",
    "prediction_is_weekend",
]

In [ ]:
stable_reactivation_categorical_features = [
    "last_order_region",
]

In [ ]:
stable_reactivation_features = (
    stable_reactivation_numeric_features
    +
    stable_reactivation_categorical_features
)

In [ ]:
# ================================================================
# 33.13 — STABLE-BEHAVIOUR PREPROCESSOR
# ================================================================

stable_reactivation_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            stable_reactivation_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="first"
                        )
                    ),
                ]
            ),
            stable_reactivation_categorical_features,
        ),
    ]
)

### 33.14 — Train Stable-Behaviour Logistic

In [ ]:
# ================================================================
# 33.14 — STABLE-BEHAVIOUR LOGISTIC
# ================================================================

stable_reactivation_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            stable_reactivation_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


stable_reactivation_logistic_model.fit(
    X_reactivation_train[
        stable_reactivation_features
    ],
    y_reactivation_train
)

In [ ]:
validation_stable_reactivation_probabilities = (
    stable_reactivation_logistic_model
    .predict_proba(
        X_reactivation_validation[
            stable_reactivation_features
        ]
    )[:, 1]
)

In [ ]:
stable_reactivation_logistic_results = (
    evaluate_reactivation_model(
        model_name="Stable-Behaviour Logistic Regression",
        y_true=y_reactivation_validation,
        probabilities=validation_stable_reactivation_probabilities,
    )
)


display(
    stable_reactivation_logistic_results
)

### 33.15 — Compare all initial models

In [ ]:
# ================================================================
# 33.15 — INITIAL REACTIVATION MODEL COMPARISON
# ================================================================

step33_model_comparison = (
    pd.concat(
        [
            prevalence_baseline_results,
            single_feature_reactivation_results,
            full_reactivation_logistic_results,
            stable_reactivation_logistic_results,
        ],
        ignore_index=True
    )
)


step33_model_comparison = (
    step33_model_comparison
    .sort_values(
        [
            "PR-AUC",
            "Lift @ 10%",
            "Lift @ 20%",
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    step33_model_comparison
)

### 33.16 — Prevalence-adjusted PR-AUC

Because validation prevalence is different from Train, let's build this correctly from the beginning.

This helps answer:
> How much better is PR-AUC than simply knowing the prevalence?

In [ ]:
# ================================================================
# 33.16 — PR-AUC RELATIVE TO VALIDATION PREVALENCE
# ================================================================

validation_reactivation_prevalence = (
    y_reactivation_validation.mean()
)


step33_model_comparison[
    "PR-AUC / Prevalence"
] = (
    step33_model_comparison[
        "PR-AUC"
    ]
    /
    validation_reactivation_prevalence
)


step33_model_comparison[
    "Normalised PR-AUC"
] = (
    (
        step33_model_comparison[
            "PR-AUC"
        ]
        -
        validation_reactivation_prevalence
    )
    /
    (
        1
        -
        validation_reactivation_prevalence
    )
)


display(
    step33_model_comparison
)

In [ ]:
# ================================================================
# 33.17 — FULL MODEL DECILE PERFORMANCE
# ================================================================

full_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_full_reactivation_probabilities,
        n_deciles=10
    )
)


display(
    full_reactivation_deciles
)

In [ ]:
# ================================================================
# 33.18 — STABLE MODEL DECILE PERFORMANCE
# ================================================================

stable_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_stable_reactivation_probabilities,
        n_deciles=10
    )
)


display(
    stable_reactivation_deciles
)

In [ ]:
# ================================================================
# VISUAL 33.1 — INITIAL PR-AUC COMPARISON
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=step33_model_comparison,
    y="Model",
    x="PR-AUC"
)


plt.axvline(
    validation_reactivation_prevalence,
    linestyle="--",
    label=(
        f"Validation prevalence "
        f"({validation_reactivation_prevalence:.3f})"
    )
)


plt.title(
    "Reactivation Model — Validation PR-AUC"
)

plt.xlabel(
    "PR-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 33.2 — TOP-20% LIFT
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=step33_model_comparison,
    y="Model",
    x="Lift @ 20%"
)


plt.axvline(
    1.0,
    linestyle="--",
    label="Random ranking"
)


plt.title(
    "Reactivation Model — Validation Lift at Top 20%"
)

plt.xlabel(
    "Lift @ 20%"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 33.3 — FULL LOGISTIC DECILES
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=full_reactivation_deciles,
    x="score_decile",
    y="actual_reactivation_rate_pct"
)


plt.axhline(
    validation_reactivation_prevalence
    *
    100,
    linestyle="--",
    label=(
        f"Validation average "
        f"({validation_reactivation_prevalence * 100:.1f}%)"
    )
)


plt.title(
    "Full-History Logistic — Reactivation Rate by Score Decile"
)

plt.xlabel(
    "Score Decile — 1 = Highest Predicted Reactivation Propensity"
)

plt.ylabel(
    "Observed Reactivation Rate (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 33.19 — FULL LOGISTIC COEFFICIENTS
# ================================================================

full_reactivation_fitted_preprocessor = (
    full_reactivation_logistic_model
    .named_steps[
        "preprocessing"
    ]
)


full_reactivation_classifier = (
    full_reactivation_logistic_model
    .named_steps[
        "model"
    ]
)


full_reactivation_feature_names = (
    full_reactivation_fitted_preprocessor
    .get_feature_names_out()
)


full_reactivation_coefficients = (
    full_reactivation_classifier
    .coef_[0]
)

In [ ]:
full_reactivation_coefficient_table = pd.DataFrame(
    {
        "feature":
            full_reactivation_feature_names,

        "coefficient":
            full_reactivation_coefficients,
    }
)


full_reactivation_coefficient_table[
    "odds_ratio"
] = np.exp(
    full_reactivation_coefficient_table[
        "coefficient"
    ]
)


full_reactivation_coefficient_table[
    "absolute_coefficient"
] = (
    full_reactivation_coefficient_table[
        "coefficient"
    ]
    .abs()
)


full_reactivation_coefficient_table[
    "direction"
] = np.where(
    full_reactivation_coefficient_table[
        "coefficient"
    ] > 0,

    "Higher reactivation propensity",

    np.where(
        full_reactivation_coefficient_table[
            "coefficient"
        ] < 0,

        "Lower reactivation propensity",

        "Neutral"
    )
)


full_reactivation_coefficient_table[
    "clean_feature"
] = (
    full_reactivation_coefficient_table[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)

In [ ]:
full_reactivation_coefficient_table = (
    full_reactivation_coefficient_table
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    full_reactivation_coefficient_table.head(30)
)

In [ ]:
# ================================================================
# VISUAL 33.4 — STRONGEST FULL-MODEL ASSOCIATIONS
# ================================================================

top_reactivation_coefficients = (
    full_reactivation_coefficient_table
    .head(20)
    .sort_values(
        "coefficient"
    )
)


plt.figure(
    figsize=(11, 9)
)


sns.barplot(
    data=top_reactivation_coefficients,
    y="clean_feature",
    x="coefficient"
)


plt.axvline(
    0,
    linewidth=1
)


plt.title(
    "Strongest Predictive Associations with 90-Day Reactivation"
)

plt.xlabel(
    "Logistic Coefficient"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 33.20 — SCORE CORRELATION BETWEEN LOGISTIC SPECIFICATIONS
# ================================================================

score_comparison = pd.DataFrame(
    {
        "full_history_score":
            validation_full_reactivation_probabilities,

        "stable_behaviour_score":
            validation_stable_reactivation_probabilities,
    }
)


score_correlation = (
    score_comparison
    .corr(
        method="spearman"
    )
    .loc[
        "full_history_score",
        "stable_behaviour_score"
    ]
)


print(
    f"Spearman score correlation: "
    f"{score_correlation:.4f}"
)

In [ ]:
# ================================================================
# 33.21 — HIGH / LOW PROPENSITY SEGMENT SUMMARY
# ================================================================

def propensity_segment_summary(
    y_true,
    probabilities,
    model_name
):

    scored = pd.DataFrame(
        {
            "actual":
                np.asarray(
                    y_true
                ),

            "score":
                np.asarray(
                    probabilities
                ),
        }
    )


    top10_threshold = (
        scored[
            "score"
        ]
        .quantile(
            0.90
        )
    )


    top20_threshold = (
        scored[
            "score"
        ]
        .quantile(
            0.80
        )
    )


    bottom20_threshold = (
        scored[
            "score"
        ]
        .quantile(
            0.20
        )
    )


    groups = []


    definitions = {
        "Top 10%":
            scored[
                "score"
            ]
            >=
            top10_threshold,

        "Top 20%":
            scored[
                "score"
            ]
            >=
            top20_threshold,

        "Bottom 20%":
            scored[
                "score"
            ]
            <=
            bottom20_threshold,
    }


    overall_rate = (
        scored[
            "actual"
        ]
        .mean()
    )


    for segment, mask in definitions.items():

        segment_data = (
            scored.loc[
                mask
            ]
        )


        segment_rate = (
            segment_data[
                "actual"
            ]
            .mean()
        )


        groups.append(
            {
                "Model":
                    model_name,

                "Segment":
                    segment,

                "Episodes":
                    len(
                        segment_data
                    ),

                "Observed Reactivation Rate":
                    segment_rate,

                "Observed Reactivation Rate (%)":
                    segment_rate
                    * 100,

                "Lift vs Overall":
                    segment_rate
                    /
                    overall_rate,
            }
        )


    return pd.DataFrame(
        groups
    )

In [ ]:
full_propensity_segments = (
    propensity_segment_summary(
        y_true=y_reactivation_validation,
        probabilities=validation_full_reactivation_probabilities,
        model_name="Full-History Logistic"
    )
)


stable_propensity_segments = (
    propensity_segment_summary(
        y_true=y_reactivation_validation,
        probabilities=validation_stable_reactivation_probabilities,
        model_name="Stable-Behaviour Logistic"
    )
)


reactivation_propensity_segments = (
    pd.concat(
        [
            full_propensity_segments,
            stable_propensity_segments,
        ],
        ignore_index=True
    )
)


display(
    reactivation_propensity_segments
)

In [ ]:
# ================================================================
# 33.22 — STEP 33 VALIDATION
# ================================================================

assert (
    len(
        validation_full_reactivation_probabilities
    )
    ==
    len(
        y_reactivation_validation
    )
)


assert (
    len(
        validation_stable_reactivation_probabilities
    )
    ==
    len(
        y_reactivation_validation
    )
)


assert np.all(
    (
        validation_full_reactivation_probabilities
        >= 0
    )
    &
    (
        validation_full_reactivation_probabilities
        <= 1
    )
)


assert np.all(
    (
        validation_stable_reactivation_probabilities
        >= 0
    )
    &
    (
        validation_stable_reactivation_probabilities
        <= 1
    )
)


assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Step 33 complete."
)

print(
    "✓ Prevalence baseline established."
)

print(
    "✓ Historical-order baseline established."
)

print(
    "✓ Full-history Logistic Regression trained."
)

print(
    "✓ Stable-behaviour Logistic challenger trained."
)

print(
    "✓ Validation ranking and lift calculated."
)

print(
    "✓ Reactivation Test population remains unopened."
)

### Step 33 Summary — Initial Reactivation Models

Step 33 established the first predictive benchmarks for 90-day customer reactivation.

Four models were evaluated on the later chronological Validation population:

1. Training Prevalence Baseline;
2. Historical Order Count Baseline;
3. Full-History Logistic Regression;
4. Stable-Behaviour Logistic Regression.

The final Test population remained untouched.

### Reactivation Is Meaningfully Predictable

The Full-History Logistic Regression achieved the strongest overall validation discrimination.

Its performance was approximately:

- **PR-AUC: 0.564**
- **ROC-AUC: 0.704**
- **Brier score: 0.208**
- **Lift@10%: 1.60×**
- **Lift@20%: 1.69×**
- **Capture@10%: 16.16%**
- **Capture@20%: 34.01%**

The validation reactivation prevalence was approximately **37.0%**.

The naive prevalence baseline therefore produced a PR-AUC of approximately **0.370** and ROC-AUC of **0.50**.

The full model materially improves on both benchmarks, demonstrating that customer history available at the 90-day inactivity threshold contains substantial information about subsequent reactivation.

### Historical Order Count Is an Exceptionally Strong Single Predictor

The Historical Order Count Baseline also performed strongly.

Using only one predictor, it achieved approximately:

- **PR-AUC: 0.538**
- **ROC-AUC: 0.684**
- **Lift@10%: 1.83×**
- **Lift@20%: 1.71×**

The one-feature model therefore produced stronger top-decile and top-20% lift than the full multivariable Logistic Regression, despite weaker overall PR-AUC and ROC-AUC.

This confirms that historical relationship depth is one of the dominant predictive signals in the reactivation problem.

The single-feature model should therefore remain a serious challenger rather than being discarded as a simple baseline.

### Full-History Logistic Provides Stronger Overall Separation

Although the Historical Order Count model provides exceptional top-of-ranking lift, the Full-History Logistic Regression provides stronger overall discrimination across the complete validation population.

The full model separates customer episodes into substantially different reactivation-risk groups.

The highest-scored 20% of inactivity episodes recorded an observed reactivation rate of approximately **62.7%**, compared with an overall validation rate of approximately **37.0%**.

The lowest-scored 20% recorded only approximately **18.6% reactivation**.

This creates an approximately **44-percentage-point difference** between the model's high- and low-propensity customer groups.

The model is therefore useful at both ends of the ranking.

### Commercial Interpretation

High-propensity customers may be customers who are relatively likely to return naturally.

These customers may not require expensive reactivation incentives.

Low-propensity customers may require stronger intervention if the business wishes to recover them.

However, propensity should not be confused with treatment responsiveness.

The current model predicts natural reactivation probability.

It does not establish which customers will change behaviour because of marketing or retention intervention.

Randomised experimentation would ultimately be required to answer that question.

### Stable-Behaviour Features Alone Were Not Sufficient

A reduced specification emphasising lower-drift behavioural ratios and recent-order characteristics achieved approximately:

- **PR-AUC: 0.508**
- **ROC-AUC: 0.664**
- **Lift@20%: 1.56×**

This remains materially better than the prevalence baseline but underperforms the full-history model.

The result indicates that cumulative relationship-depth features contain genuine predictive information despite their substantial temporal drift.

They should therefore not be excluded simply because their distributions change over time.

### Coefficient Interpretation Requires Caution

The full Logistic Regression includes several highly related customer-history variables.

Examples include:

- historical order count and its logarithmic transformation;
- historical revenue and log historical revenue;
- prior inactivity episode count;
- prior reactivation count;
- prior-reactivation indicators;
- tenure;
- and frequency measures.

Because these variables overlap conceptually, individual coefficients may be unstable or appear contradictory.

The current coefficient table should therefore be interpreted primarily as evidence of predictive association rather than precise structural effects.

A simplified and redundancy-aware logistic specification should be evaluated before final coefficient interpretation.

### Initial Modelling Conclusion

The Full-History Logistic Regression is the current overall champion because it provides the strongest PR-AUC, ROC-AUC and broad customer separation.

However, Historical Order Count alone remains an important challenger because it produces exceptional top-of-ranking lift with extreme simplicity.

The next modelling stage should determine whether non-linear models can improve on these results and whether a cleaner, redundancy-reduced Logistic Regression can preserve performance while improving interpretability.


## Step 34 — Non-Linear Models and Redundancy-Reduced Logistic Challenger

Step 33 established that customer reactivation is meaningfully predictable.

The Full-History Logistic Regression produced the strongest overall validation discrimination, while Historical Order Count alone produced exceptionally strong top-of-ranking lift.

Step 34 therefore evaluates whether:

- non-linear models can improve predictive discrimination or ranking;
- a simplified Logistic Regression can preserve performance while reducing feature redundancy;
- and the strong relationship-depth signal remains dominant after model complexity increases.

The models evaluated in this stage are:

1. Historical Order Count Baseline;
2. Full-History Logistic Regression;
3. Redundancy-Reduced Logistic Regression;
4. Random Forest;
5. Gradient Boosting;
6. HistGradientBoosting.

All models are trained on the same chronological Train population and evaluated on the same later Validation population.

The final Test population remains untouched.

### Model Selection Principle

Model selection will prioritise:

- PR-AUC;
- ROC-AUC;
- Lift@10%;
- Lift@20%;
- repeater capture;
- calibration quality;
- temporal robustness;
- interpretability;
- and implementation simplicity.

A more complex model will only replace Logistic Regression if it produces a meaningful improvement rather than a marginal metric gain.

In [ ]:
# ================================================================
# 34.2 — NON-LINEAR MODEL IMPORTS
# ================================================================

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

In [ ]:
# ================================================================
# 34.3 — TEST-SET GOVERNANCE
# ================================================================

assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Reactivation Test population remains unopened."
)

print(
    "Step 34 uses Train + Validation only."
)

### 34.4 — Define redundancy-reduced feature philosophy

We want to remove obvious duplication.

The full model currently contains combinations like:
- raw revenue + log revenue;
- raw units + log units;
- order count + log order count;
- prior inactivity count + binary prior-inactivity indicator;
- prior reactivation count + prior-reactivation rate + prior-success indicator.

Those may help prediction, but they make coefficient interpretation difficult.

The simplified model should keep one representative variable from each behavioural concept.

In [ ]:
# ================================================================
# 34.5 — REDUNDANCY-REDUCED FEATURE SET
# ================================================================

reduced_reactivation_numeric_features = [
    # Relationship depth
    "log1p_historical_order_count",
    "log1p_historical_revenue",
    "tenure_at_prediction_days",

    # Purchase cadence
    "historical_average_gap_days",
    "historical_max_gap_days",
    "orders_per_30_tenure_days",
    "inactivity_to_average_gap_ratio",

    # Recent transaction
    "last_order_value",
    "last_order_units",
    "last_order_product_count",
    "last_order_average_unit_price",

    # Relationship economics
    "historical_average_order_value",
    "last_to_historical_aov_ratio",
    "revenue_per_30_tenure_days",

    # Prior inactivity/reactivation
    "prior_inactivity_episode_count",
    "prior_reactivation_rate",
    "has_prior_successful_reactivation",

    # History availability
    "has_prior_purchase_gap",

    # Calendar
    "prediction_month",
    "prediction_weekday",
    "prediction_is_weekend",
]


reduced_reactivation_categorical_features = [
    "last_order_region",
]


reduced_reactivation_features = (
    reduced_reactivation_numeric_features
    +
    reduced_reactivation_categorical_features
)

In [ ]:
# ================================================================
# 34.6 — REDUCED LOGISTIC PREPROCESSOR
# ================================================================

reduced_reactivation_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    ),

                    (
                        "scaler",
                        StandardScaler()
                    ),
                ]
            ),
            reduced_reactivation_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="first"
                        )
                    ),
                ]
            ),
            reduced_reactivation_categorical_features,
        ),
    ]
)

In [ ]:
# ================================================================
# 34.7 — REDUNDANCY-REDUCED LOGISTIC
# ================================================================

reduced_reactivation_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            reduced_reactivation_preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        ),
    ]
)


reduced_reactivation_logistic_model.fit(
    X_reactivation_train[
        reduced_reactivation_features
    ],
    y_reactivation_train
)

In [ ]:
validation_reduced_reactivation_probabilities = (
    reduced_reactivation_logistic_model
    .predict_proba(
        X_reactivation_validation[
            reduced_reactivation_features
        ]
    )[:, 1]
)

In [ ]:
reduced_reactivation_logistic_results = (
    evaluate_reactivation_model(
        model_name="Reduced Logistic Regression",
        y_true=y_reactivation_validation,
        probabilities=validation_reduced_reactivation_probabilities,
    )
)


display(
    reduced_reactivation_logistic_results
)

### 34.8 — Tree-model preprocessing
Tree models do not require scaling.

We still need:
- median numeric imputation;
- categorical imputation;
- dense one-hot encoding.

In [ ]:
# ================================================================
# 34.8 — TREE MODEL PREPROCESSOR
# ================================================================

tree_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        )
    ]
)

In [ ]:
tree_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ),
    ]
)

In [ ]:
tree_reactivation_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_transformer,
            reactivation_numeric_features,
        ),

        (
            "categorical",
            tree_categorical_transformer,
            reactivation_categorical_features,
        ),
    ]
)

### 34.9 — Random Forest
We want enough complexity to capture interactions without allowing extremely deep trees.

In [ ]:
# ================================================================
# 34.9 — RANDOM FOREST
# ================================================================

random_forest_reactivation_model = Pipeline(
    steps=[
        (
            "preprocessing",
            tree_reactivation_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=15,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)

In [ ]:
random_forest_reactivation_model.fit(
    X_reactivation_train,
    y_reactivation_train
)

In [ ]:
validation_random_forest_probabilities = (
    random_forest_reactivation_model
    .predict_proba(
        X_reactivation_validation
    )[:, 1]
)

In [ ]:
random_forest_reactivation_results = (
    evaluate_reactivation_model(
        model_name="Random Forest",
        y_true=y_reactivation_validation,
        probabilities=validation_random_forest_probabilities,
    )
)


display(
    random_forest_reactivation_results
)

### 34.10 — Gradient Boosting

In [ ]:
# ================================================================
# 34.10 — GRADIENT BOOSTING
# ================================================================

gradient_boosting_reactivation_model = Pipeline(
    steps=[
        (
            "preprocessing",
            tree_reactivation_preprocessor
        ),

        (
            "model",
            GradientBoostingClassifier(
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                min_samples_leaf=15,
                random_state=42
            )
        ),
    ]
)

In [ ]:
gradient_boosting_reactivation_model.fit(
    X_reactivation_train,
    y_reactivation_train
)

In [ ]:
validation_gradient_boosting_probabilities = (
    gradient_boosting_reactivation_model
    .predict_proba(
        X_reactivation_validation
    )[:, 1]
)

In [ ]:
gradient_boosting_reactivation_results = (
    evaluate_reactivation_model(
        model_name="Gradient Boosting",
        y_true=y_reactivation_validation,
        probabilities=validation_gradient_boosting_probabilities,
    )
)


display(
    gradient_boosting_reactivation_results
)

### 34.11 — HistGradientBoosting

In [ ]:
# ================================================================
# 34.11 — HISTGRADIENTBOOSTING
# ================================================================

hist_gradient_reactivation_model = Pipeline(
    steps=[
        (
            "preprocessing",
            tree_reactivation_preprocessor
        ),

        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=20,
                l2_regularization=1.0,
                random_state=42
            )
        ),
    ]
)

In [ ]:
hist_gradient_reactivation_model.fit(
    X_reactivation_train,
    y_reactivation_train
)

In [ ]:
validation_hist_gradient_probabilities = (
    hist_gradient_reactivation_model
    .predict_proba(
        X_reactivation_validation
    )[:, 1]
)

In [ ]:
hist_gradient_reactivation_results = (
    evaluate_reactivation_model(
        model_name="HistGradientBoosting",
        y_true=y_reactivation_validation,
        probabilities=validation_hist_gradient_probabilities,
    )
)


display(
    hist_gradient_reactivation_results
)

### 34.12 — Comprehensive model comparison
Now compare:
- Prevalence
- Order Count
- Stable Logistic
- Reduced Logistic
- Full Logistic
- Random Forest
- Gradient Boosting
- HistGradientBoosting

In [ ]:
# ================================================================
# 34.12 — COMPLETE VALIDATION MODEL COMPARISON
# ================================================================

step34_model_comparison = (
    pd.concat(
        [
            prevalence_baseline_results,
            single_feature_reactivation_results,
            stable_reactivation_logistic_results,
            reduced_reactivation_logistic_results,
            full_reactivation_logistic_results,
            random_forest_reactivation_results,
            gradient_boosting_reactivation_results,
            hist_gradient_reactivation_results,
        ],
        ignore_index=True
    )
)

In [ ]:
step34_model_comparison = (
    step34_model_comparison
    .sort_values(
        [
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 20%"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    step34_model_comparison
)

### 34.13 — Add prevalence-adjusted PR-AUC

In [ ]:
# ================================================================
# 34.13 — PREVALENCE-ADJUSTED PR-AUC
# ================================================================

step34_model_comparison[
    "PR-AUC / Prevalence"
] = (
    step34_model_comparison[
        "PR-AUC"
    ]
    /
    validation_reactivation_prevalence
)


step34_model_comparison[
    "Normalised PR-AUC"
] = (
    (
        step34_model_comparison[
            "PR-AUC"
        ]
        -
        validation_reactivation_prevalence
    )
    /
    (
        1
        -
        validation_reactivation_prevalence
    )
)


display(
    step34_model_comparison
)

In [ ]:
# ================================================================
# VISUAL 34.1 — PR-AUC
# ================================================================

plt.figure(
    figsize=(12, 7)
)


sns.barplot(
    data=step34_model_comparison,
    y="Model",
    x="PR-AUC"
)


plt.axvline(
    validation_reactivation_prevalence,
    linestyle="--",
    label=(
        f"Validation prevalence "
        f"({validation_reactivation_prevalence:.3f})"
    )
)


plt.title(
    "Reactivation Models — Validation PR-AUC"
)

plt.xlabel(
    "PR-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 34.2 — ROC-AUC
# ================================================================

plt.figure(
    figsize=(12, 7)
)


sns.barplot(
    data=step34_model_comparison,
    y="Model",
    x="ROC-AUC"
)


plt.axvline(
    0.50,
    linestyle="--",
    label="Random ranking"
)


plt.title(
    "Reactivation Models — Validation ROC-AUC"
)

plt.xlabel(
    "ROC-AUC"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 34.3 — TOP-10% LIFT
# ================================================================

plt.figure(
    figsize=(12, 7)
)


sns.barplot(
    data=step34_model_comparison,
    y="Model",
    x="Lift @ 10%"
)


plt.axvline(
    1,
    linestyle="--",
    label="Random targeting"
)


plt.title(
    "Reactivation Models — Top-Decile Lift"
)

plt.xlabel(
    "Lift @ 10%"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 34.4 — TOP-20% LIFT
# ================================================================

plt.figure(
    figsize=(12, 7)
)


sns.barplot(
    data=step34_model_comparison,
    y="Model",
    x="Lift @ 20%"
)


plt.axvline(
    1,
    linestyle="--",
    label="Random targeting"
)


plt.title(
    "Reactivation Models — Top-20% Lift"
)

plt.xlabel(
    "Lift @ 20%"
)

plt.ylabel(
    ""
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 34.14 — VALIDATION DECILES
# ================================================================

reduced_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_reduced_reactivation_probabilities
    )
)


random_forest_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_random_forest_probabilities
    )
)


gradient_boosting_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_gradient_boosting_probabilities
    )
)


hist_gradient_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_hist_gradient_probabilities
    )
)

In [ ]:
# ================================================================
# 34.15 — COMMERCIAL SEGMENT COMPARISON
# ================================================================

segment_tables = [
    propensity_segment_summary(
        y_reactivation_validation,
        validation_single_feature_probabilities,
        "Historical Order Count"
    ),

    propensity_segment_summary(
        y_reactivation_validation,
        validation_full_reactivation_probabilities,
        "Full-History Logistic"
    ),

    propensity_segment_summary(
        y_reactivation_validation,
        validation_reduced_reactivation_probabilities,
        "Reduced Logistic"
    ),

    propensity_segment_summary(
        y_reactivation_validation,
        validation_random_forest_probabilities,
        "Random Forest"
    ),

    propensity_segment_summary(
        y_reactivation_validation,
        validation_gradient_boosting_probabilities,
        "Gradient Boosting"
    ),

    propensity_segment_summary(
        y_reactivation_validation,
        validation_hist_gradient_probabilities,
        "HistGradientBoosting"
    ),
]


step34_segment_comparison = pd.concat(
    segment_tables,
    ignore_index=True
)


display(
    step34_segment_comparison
)

In [ ]:
# ================================================================
# 34.16 — MODEL-SELECTION SCORECARD
# ================================================================

step34_selection_scorecard = (
    step34_model_comparison[
        [
            "Model",
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
            "PR-AUC / Prevalence",
            "Normalised PR-AUC",
        ]
    ]
    .copy()
)


display(
    step34_selection_scorecard
)

In [ ]:
# ================================================================
# 34.17 — REDUCED VS FULL LOGISTIC DELTAS
# ================================================================

full_row = (
    full_reactivation_logistic_results
    .iloc[0]
)


reduced_row = (
    reduced_reactivation_logistic_results
    .iloc[0]
)


reduced_vs_full = pd.DataFrame(
    {
        "Metric": [
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ],

        "Full Logistic": [
            full_row[
                "PR-AUC"
            ],
            full_row[
                "ROC-AUC"
            ],
            full_row[
                "Brier Score"
            ],
            full_row[
                "Lift @ 10%"
            ],
            full_row[
                "Lift @ 20%"
            ],
            full_row[
                "Capture @ 10%"
            ],
            full_row[
                "Capture @ 20%"
            ],
        ],

        "Reduced Logistic": [
            reduced_row[
                "PR-AUC"
            ],
            reduced_row[
                "ROC-AUC"
            ],
            reduced_row[
                "Brier Score"
            ],
            reduced_row[
                "Lift @ 10%"
            ],
            reduced_row[
                "Lift @ 20%"
            ],
            reduced_row[
                "Capture @ 10%"
            ],
            reduced_row[
                "Capture @ 20%"
            ],
        ],
    }
)

In [ ]:
reduced_vs_full[
    "Reduced - Full"
] = (
    reduced_vs_full[
        "Reduced Logistic"
    ]
    -
    reduced_vs_full[
        "Full Logistic"
    ]
)


display(
    reduced_vs_full
)

### 34.18 — Feature importance: Random Forest
If Random Forest is competitive, inspect its feature importance.

In [ ]:
# ================================================================
# 34.18 — RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

rf_preprocessor = (
    random_forest_reactivation_model
    .named_steps[
        "preprocessing"
    ]
)


rf_classifier = (
    random_forest_reactivation_model
    .named_steps[
        "model"
    ]
)


rf_feature_names = (
    rf_preprocessor
    .get_feature_names_out()
)


rf_importances = (
    rf_classifier
    .feature_importances_
)


random_forest_feature_importance = pd.DataFrame(
    {
        "feature":
            rf_feature_names,

        "importance":
            rf_importances,
    }
)

In [ ]:
random_forest_feature_importance[
    "clean_feature"
] = (
    random_forest_feature_importance[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)

In [ ]:
random_forest_feature_importance = (
    random_forest_feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    random_forest_feature_importance.head(25)
)

In [ ]:
# ================================================================
# VISUAL 34.5 — RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

top_rf_importance = (
    random_forest_feature_importance
    .head(20)
    .sort_values(
        "importance"
    )
)


plt.figure(
    figsize=(11, 9)
)


sns.barplot(
    data=top_rf_importance,
    y="clean_feature",
    x="importance"
)


plt.title(
    "Random Forest — Most Important Reactivation Predictors"
)

plt.xlabel(
    "Feature Importance"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 34.19 — GRADIENT BOOSTING FEATURE IMPORTANCE
# ================================================================

gb_preprocessor = (
    gradient_boosting_reactivation_model
    .named_steps[
        "preprocessing"
    ]
)


gb_classifier = (
    gradient_boosting_reactivation_model
    .named_steps[
        "model"
    ]
)


gb_feature_names = (
    gb_preprocessor
    .get_feature_names_out()
)


gb_importances = (
    gb_classifier
    .feature_importances_
)


gradient_boosting_feature_importance = pd.DataFrame(
    {
        "feature":
            gb_feature_names,

        "importance":
            gb_importances,
    }
)

In [ ]:
gradient_boosting_feature_importance[
    "clean_feature"
] = (
    gradient_boosting_feature_importance[
        "feature"
    ]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
    .str.replace(
        "categorical__",
        "",
        regex=False
    )
)


gradient_boosting_feature_importance = (
    gradient_boosting_feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    gradient_boosting_feature_importance.head(25)
)

In [ ]:
# ================================================================
# 34.20 — MODEL SCORE CORRELATION
# ================================================================

reactivation_model_scores = pd.DataFrame(
    {
        "Order_Count":
            validation_single_feature_probabilities,

        "Full_Logistic":
            validation_full_reactivation_probabilities,

        "Reduced_Logistic":
            validation_reduced_reactivation_probabilities,

        "Random_Forest":
            validation_random_forest_probabilities,

        "Gradient_Boosting":
            validation_gradient_boosting_probabilities,

        "HistGradient":
            validation_hist_gradient_probabilities,
    }
)


reactivation_score_correlation = (
    reactivation_model_scores
    .corr(
        method="spearman"
    )
)


display(
    reactivation_score_correlation.round(3)
)

In [ ]:
# ================================================================
# VISUAL 34.6 — MODEL SCORE CORRELATION
# ================================================================

plt.figure(
    figsize=(9, 7)
)


sns.heatmap(
    reactivation_score_correlation,
    annot=True,
    cmap="coolwarm",
    center=0
)


plt.title(
    "Spearman Correlation Between Reactivation Model Rankings"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 34.21 — TOP-20% CUSTOMER OVERLAP
# ================================================================

def top_fraction_indices(
    probabilities,
    fraction=0.20
):

    probabilities = np.asarray(
        probabilities
    )


    n_top = max(
        1,
        int(
            np.ceil(
                len(probabilities)
                *
                fraction
            )
        )
    )


    return set(
        np.argsort(
            probabilities
        )[
            -n_top:
        ]
    )

In [ ]:
top20_sets = {
    "Order_Count":
        top_fraction_indices(
            validation_single_feature_probabilities,
            0.20
        ),

    "Full_Logistic":
        top_fraction_indices(
            validation_full_reactivation_probabilities,
            0.20
        ),

    "Reduced_Logistic":
        top_fraction_indices(
            validation_reduced_reactivation_probabilities,
            0.20
        ),

    "Random_Forest":
        top_fraction_indices(
            validation_random_forest_probabilities,
            0.20
        ),

    "Gradient_Boosting":
        top_fraction_indices(
            validation_gradient_boosting_probabilities,
            0.20
        ),

    "HistGradient":
        top_fraction_indices(
            validation_hist_gradient_probabilities,
            0.20
        ),
}

In [ ]:
model_names = list(
    top20_sets.keys()
)


top20_overlap_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)


for model_a in model_names:

    for model_b in model_names:

        intersection_size = len(
            top20_sets[
                model_a
            ]
            &
            top20_sets[
                model_b
            ]
        )


        denominator = len(
            top20_sets[
                model_a
            ]
        )


        top20_overlap_matrix.loc[
            model_a,
            model_b
        ] = (
            intersection_size
            /
            denominator
        )


display(
    top20_overlap_matrix.round(3)
)

In [ ]:
# ================================================================
# 34.22 — BOTTOM-20% CUSTOMER OVERLAP
# ================================================================

def bottom_fraction_indices(
    probabilities,
    fraction=0.20
):

    probabilities = np.asarray(
        probabilities
    )


    n_bottom = max(
        1,
        int(
            np.ceil(
                len(probabilities)
                *
                fraction
            )
        )
    )


    return set(
        np.argsort(
            probabilities
        )[
            :n_bottom
        ]
    )

In [ ]:
bottom20_sets = {
    "Order_Count":
        bottom_fraction_indices(
            validation_single_feature_probabilities
        ),

    "Full_Logistic":
        bottom_fraction_indices(
            validation_full_reactivation_probabilities
        ),

    "Reduced_Logistic":
        bottom_fraction_indices(
            validation_reduced_reactivation_probabilities
        ),

    "Random_Forest":
        bottom_fraction_indices(
            validation_random_forest_probabilities
        ),

    "Gradient_Boosting":
        bottom_fraction_indices(
            validation_gradient_boosting_probabilities
        ),

    "HistGradient":
        bottom_fraction_indices(
            validation_hist_gradient_probabilities
        ),
}

In [ ]:
bottom20_overlap_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)


for model_a in model_names:

    for model_b in model_names:

        intersection_size = len(
            bottom20_sets[
                model_a
            ]
            &
            bottom20_sets[
                model_b
            ]
        )


        denominator = len(
            bottom20_sets[
                model_a
            ]
        )


        bottom20_overlap_matrix.loc[
            model_a,
            model_b
        ] = (
            intersection_size
            /
            denominator
        )


display(
    bottom20_overlap_matrix.round(3)
)

In [ ]:
# ================================================================
# 34.24 — CALIBRATION / PROBABILITY QUALITY
# ================================================================

step34_calibration_comparison = (
    step34_model_comparison[
        [
            "Model",
            "Brier Score",
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 10%",
            "Lift @ 20%",
        ]
    ]
    .sort_values(
        "Brier Score"
    )
    .reset_index(
        drop=True
    )
)


display(
    step34_calibration_comparison
)

In [ ]:
# ================================================================
# 34.25 — PRACTICAL EQUIVALENCE BAND
# ================================================================

REACTIVATION_PR_AUC_TOLERANCE = 0.005


best_validation_pr_auc = (
    step34_selection_scorecard[
        "PR-AUC"
    ]
    .max()
)


step34_selection_scorecard[
    "Within_PR_AUC_Equivalence_Band"
] = (
    step34_selection_scorecard[
        "PR-AUC"
    ]
    >=
    (
        best_validation_pr_auc
        -
        REACTIVATION_PR_AUC_TOLERANCE
    )
)


display(
    step34_selection_scorecard
)

In [ ]:
# ================================================================
# 34.26 — MODEL OBJECT REGISTER
# ================================================================

reactivation_candidate_models = {
    "Historical Order Count Baseline":
        single_feature_reactivation_model,

    "Full-History Logistic Regression":
        full_reactivation_logistic_model,

    "Reduced Logistic Regression":
        reduced_reactivation_logistic_model,

    "Random Forest":
        random_forest_reactivation_model,

    "Gradient Boosting":
        gradient_boosting_reactivation_model,

    "HistGradientBoosting":
        hist_gradient_reactivation_model,
}

In [ ]:
reactivation_candidate_probabilities = {
    "Historical Order Count Baseline":
        validation_single_feature_probabilities,

    "Full-History Logistic Regression":
        validation_full_reactivation_probabilities,

    "Reduced Logistic Regression":
        validation_reduced_reactivation_probabilities,

    "Random Forest":
        validation_random_forest_probabilities,

    "Gradient Boosting":
        validation_gradient_boosting_probabilities,

    "HistGradientBoosting":
        validation_hist_gradient_probabilities,
}

In [ ]:
# ================================================================
# 34.27 — PRELIMINARY VALIDATION LEADER
# ================================================================

preliminary_reactivation_champion = (
    step34_selection_scorecard
    .sort_values(
        [
            "PR-AUC",
            "ROC-AUC",
            "Lift @ 20%"
        ],
        ascending=False
    )
    .iloc[0]
)


print(
    "Current validation leader:"
)

print(
    preliminary_reactivation_champion[
        "Model"
    ]
)

In [ ]:
# ================================================================
# 34.28 — STEP 34 COMPLETION CHECK
# ================================================================

for probabilities in [
    validation_random_forest_probabilities,
    validation_gradient_boosting_probabilities,
    validation_hist_gradient_probabilities,
    validation_reduced_reactivation_probabilities,
]:

    assert (
        len(
            probabilities
        )
        ==
        len(
            y_reactivation_validation
        )
    )


    assert np.all(
        (
            probabilities
            >= 0
        )
        &
        (
            probabilities
            <= 1
        )
    )


assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Step 34 complete."
)

print(
    "✓ Reduced Logistic evaluated."
)

print(
    "✓ Random Forest evaluated."
)

print(
    "✓ Gradient Boosting evaluated."
)

print(
    "✓ HistGradientBoosting evaluated."
)

print(
    "✓ Model rankings and segment performance compared."
)

print(
    "✓ Reactivation Test population remains unopened."
)

### Step 34 Summary — Non-Linear Models and Simplified Logistic Regression

Step 34 evaluated whether non-linear models could improve prediction of customer reactivation and whether the original Full-History Logistic Regression could be simplified without sacrificing predictive performance.

The final Test population remained untouched throughout this stage.

### Random Forest Became the Strongest Validation Model

Random Forest produced the strongest overall validation performance:

- **PR-AUC: 0.607**
- **ROC-AUC: 0.721**
- **Brier score: 0.203**
- **Lift@10%: 2.00×**
- **Lift@20%: 1.81×**
- **Capture@10%: 20.20%**
- **Capture@20%: 36.36%**

These results materially exceed the training-prevalence baseline and improve on every major validation metric produced by the Logistic Regression specifications.

The highest-ranked 10% of inactivity episodes recorded an observed reactivation rate of approximately **74.1%**, compared with an overall validation rate of approximately **37.0%**.

The highest-ranked 20% recorded approximately **67.1% reactivation**.

The lowest-ranked 20% recorded only approximately **18.6% reactivation**.

This demonstrates strong customer segmentation at both ends of the score distribution.

### Model Complexity Added Genuine Predictive Value

Unlike the repeat-purchase problem examined in Notebook 2, non-linear modelling materially improved validation performance.

Random Forest increased PR-AUC from approximately **0.584 for the strongest Logistic challenger to 0.607**, while increasing ROC-AUC from approximately **0.708 to 0.721**.

Top-decile lift also increased from approximately **1.77× to 2.00×**.

The gain therefore appears across overall discrimination, commercial lift and probability error rather than a single isolated metric.

Random Forest is consequently the provisional reactivation-model champion.

### Reducing Logistic Redundancy Improved Performance

The Redundancy-Reduced Logistic Regression materially improved on the original Full-History Logistic specification.

Relative to the full Logistic model:

- PR-AUC improved from approximately **0.564 to 0.584**
- ROC-AUC improved from approximately **0.704 to 0.708**
- Brier score improved from approximately **0.208 to 0.205**
- Lift@10% improved from approximately **1.60× to 1.77×**
- Lift@20% improved from approximately **1.69× to 1.71×**

The result confirms that the original full feature set contained unnecessary overlapping information.

The Reduced Logistic Regression should therefore replace the original Full-History Logistic as the primary interpretable challenger.

### Historical Order Count Remains a Powerful Benchmark

Historical Order Count alone continued to provide strong ranking performance.

The single-feature model achieved:

- **PR-AUC: 0.538**
- **ROC-AUC: 0.684**
- **Lift@10%: 1.83×**
- **Lift@20%: 1.71×**

This confirms that customer relationship depth is a dominant driver of natural reactivation propensity.

More complex models improve on this signal, but the single-feature model demonstrates that a large proportion of predictive power originates from how established the customer relationship was before inactivity.

### Candidate Models Are Related but Not Identical

Model-score correlations show that the various approaches rank customers in broadly similar but not identical ways.

The Reduced and Full Logistic specifications are almost identical in ranking behaviour, with a Spearman correlation of approximately **0.99**.

Random Forest correlates approximately **0.85** with the Logistic models.

This indicates that Random Forest captures additional nonlinear structure while remaining broadly aligned with the core relationship-depth signal.

### Practical-Equivalence Decision

A predetermined PR-AUC practical-equivalence tolerance of **0.005** was used.

Random Forest achieved PR-AUC of approximately **0.607**, while the nearest challenger, Reduced Logistic Regression, achieved approximately **0.584**.

Only Random Forest falls within the best-model equivalence band.

The Random Forest improvement is therefore sufficiently large to justify continued evaluation despite its additional complexity.

### Step 34 Model Decision

**Random Forest becomes the provisional champion.**

The **Reduced Logistic Regression** remains the principal interpretable challenger.

The model is not yet frozen.

Before final selection, the next stage must assess:

- performance stability through time;
- feature importance and dependence on drifting variables;
- probability calibration;
- ranking robustness;
- and whether the validation advantage remains credible across historical periods.

The final temporal Test population remains unopened.



## Step 35 — Temporal Stability, Feature Dependence and Calibration

Random Forest emerged from Step 34 as the strongest validation model.

However, the reactivation population changes substantially through time.

Before freezing a final model, Step 35 therefore tests whether the Random Forest advantage is robust rather than specific to one validation period.

The analysis evaluates:

- temporal performance across historical folds;
- stability relative to Reduced Logistic Regression;
- dependence on customer-history features that drift through time;
- probability calibration;
- validation uncertainty;
- and ranking consistency.

Random Forest will only be frozen if its predictive advantage remains sufficiently credible after these diagnostics.

The final Test population remains unopened.

In [ ]:
# ================================================================
# 35.2 — TEST GOVERNANCE
# ================================================================

assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)

print(
    "✓ Reactivation Test population remains unopened."
)

In [ ]:
# ================================================================
# 35.3 — CHRONOLOGICALLY ORDER TRAINING POPULATION
# ================================================================

reactivation_train_ordered = (
    train_reactivation_data
    .sort_values(
        [
            "prediction_date",
            "customer"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Training episodes: "
    f"{len(reactivation_train_ordered):,}"
)

print(
    f"Training period: "
    f"{reactivation_train_ordered['prediction_period'].min()} "
    f"to "
    f"{reactivation_train_ordered['prediction_period'].max()}"
)

In [ ]:
# ================================================================
# 35.4 — CREATE MONTH-BASED EXPANDING FOLDS
# ================================================================

train_periods = sorted(
    reactivation_train_ordered[
        "prediction_period"
    ]
    .unique()
)


print(
    "Available training periods:"
)

print(
    train_periods
)

In [ ]:
def create_expanding_month_folds(
    data,
    period_column,
    min_training_periods=4
):
    """
    Create expanding-window temporal folds.

    Each fold trains on all months before one validation month,
    then evaluates on the next month.
    """

    periods = sorted(
        data[
            period_column
        ]
        .unique()
    )


    folds = []


    for validation_position in range(
        min_training_periods,
        len(periods)
    ):

        validation_period = (
            periods[
                validation_position
            ]
        )


        training_periods = (
            periods[
                :validation_position
            ]
        )


        train_indices = (
            data.index[
                data[
                    period_column
                ]
                .isin(
                    training_periods
                )
            ]
            .to_numpy()
        )


        validation_indices = (
            data.index[
                data[
                    period_column
                ]
                ==
                validation_period
            ]
            .to_numpy()
        )


        if (
            len(train_indices) > 0
            and
            len(validation_indices) > 0
        ):

            folds.append(
                {
                    "validation_period":
                        validation_period,

                    "train_indices":
                        train_indices,

                    "validation_indices":
                        validation_indices,
                }
            )


    return folds

In [ ]:
reactivation_temporal_folds = (
    create_expanding_month_folds(
        data=reactivation_train_ordered,
        period_column="prediction_period",
        min_training_periods=4
    )
)


print(
    f"Temporal folds created: "
    f"{len(reactivation_temporal_folds)}"
)

In [ ]:
# ================================================================
# 35.5 — TEMPORAL MODEL COMPARISON
# ================================================================

temporal_model_records = []


for fold_number, fold in enumerate(
    reactivation_temporal_folds,
    start=1
):

    fold_train = (
        reactivation_train_ordered.loc[
            fold[
                "train_indices"
            ]
        ]
    )


    fold_validation = (
        reactivation_train_ordered.loc[
            fold[
                "validation_indices"
            ]
        ]
    )


    y_fold_train = (
        fold_train[
            reactivation_target
        ]
    )


    y_fold_validation = (
        fold_validation[
            reactivation_target
        ]
    )


    # ------------------------------------------------------------
    # Skip pathological single-class validation months
    # ------------------------------------------------------------

    if (
        y_fold_validation.nunique()
        < 2
    ):
        continue


    # ------------------------------------------------------------
    # RANDOM FOREST
    # ------------------------------------------------------------

    rf_fold_model = clone(
        random_forest_reactivation_model
    )


    rf_fold_model.fit(
        fold_train[
            reactivation_model_features
        ],
        y_fold_train
    )


    rf_fold_probabilities = (
        rf_fold_model
        .predict_proba(
            fold_validation[
                reactivation_model_features
            ]
        )[:, 1]
    )


    # ------------------------------------------------------------
    # REDUCED LOGISTIC
    # ------------------------------------------------------------

    logistic_fold_model = clone(
        reduced_reactivation_logistic_model
    )


    logistic_fold_model.fit(
        fold_train[
            reduced_reactivation_features
        ],
        y_fold_train
    )


    logistic_fold_probabilities = (
        logistic_fold_model
        .predict_proba(
            fold_validation[
                reduced_reactivation_features
            ]
        )[:, 1]
    )


    # ------------------------------------------------------------
    # METRICS
    # ------------------------------------------------------------

    for (
        model_name,
        probabilities
    ) in [
        (
            "Random Forest",
            rf_fold_probabilities
        ),

        (
            "Reduced Logistic",
            logistic_fold_probabilities
        ),
    ]:

        temporal_model_records.append(
            {
                "fold":
                    fold_number,

                "validation_period":
                    str(
                        fold[
                            "validation_period"
                        ]
                    ),

                "model":
                    model_name,

                "train_episodes":
                    len(
                        fold_train
                    ),

                "validation_episodes":
                    len(
                        fold_validation
                    ),

                "validation_prevalence":
                    y_fold_validation.mean(),

                "PR_AUC":
                    average_precision_score(
                        y_fold_validation,
                        probabilities
                    ),

                "ROC_AUC":
                    roc_auc_score(
                        y_fold_validation,
                        probabilities
                    ),

                "Brier_Score":
                    brier_score_loss(
                        y_fold_validation,
                        probabilities
                    ),

                "Lift_10":
                    lift_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.10
                    ),

                "Lift_20":
                    lift_at_fraction(
                        y_fold_validation,
                        probabilities,
                        0.20
                    ),
            }
        )


temporal_reactivation_results = pd.DataFrame(
    temporal_model_records
)


display(
    temporal_reactivation_results
)

In [ ]:
# ================================================================
# 35.6 — TEMPORAL STABILITY SUMMARY
# ================================================================

temporal_reactivation_summary = (
    temporal_reactivation_results
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        folds=(
            "fold",
            "nunique"
        ),

        mean_PR_AUC=(
            "PR_AUC",
            "mean"
        ),

        std_PR_AUC=(
            "PR_AUC",
            "std"
        ),

        min_PR_AUC=(
            "PR_AUC",
            "min"
        ),

        max_PR_AUC=(
            "PR_AUC",
            "max"
        ),

        mean_ROC_AUC=(
            "ROC_AUC",
            "mean"
        ),

        std_ROC_AUC=(
            "ROC_AUC",
            "std"
        ),

        mean_Lift_10=(
            "Lift_10",
            "mean"
        ),

        min_Lift_10=(
            "Lift_10",
            "min"
        ),

        mean_Lift_20=(
            "Lift_20",
            "mean"
        ),

        min_Lift_20=(
            "Lift_20",
            "min"
        ),

        mean_Brier=(
            "Brier_Score",
            "mean"
        ),
    )
)


display(
    temporal_reactivation_summary
)

### VISUAL 35.1 — PR-AUC through time

In [ ]:
plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=temporal_reactivation_results,
    x="validation_period",
    y="PR_AUC",
    hue="model",
    marker="o"
)


plt.title(
    "Reactivation Model PR-AUC Across Historical Time"
)

plt.xlabel(
    "Validation Month"
)

plt.ylabel(
    "PR-AUC"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

### VISUAL 35.2 — ROC-AUC through time

In [ ]:
plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=temporal_reactivation_results,
    x="validation_period",
    y="ROC_AUC",
    hue="model",
    marker="o"
)


plt.axhline(
    0.50,
    linestyle="--"
)


plt.title(
    "Reactivation ROC-AUC Across Historical Time"
)

plt.xlabel(
    "Validation Month"
)

plt.ylabel(
    "ROC-AUC"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

### VISUAL 35.3 — Lift@20 through time

In [ ]:
plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=temporal_reactivation_results,
    x="validation_period",
    y="Lift_20",
    hue="model",
    marker="o"
)


plt.axhline(
    1,
    linestyle="--"
)


plt.title(
    "Reactivation Lift@20% Across Historical Time"
)

plt.xlabel(
    "Validation Month"
)

plt.ylabel(
    "Lift @ 20%"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 35.7 — FOLD-BY-FOLD MODEL WINS
# ================================================================

temporal_pivot = (
    temporal_reactivation_results
    .pivot(
        index="validation_period",
        columns="model",
        values=[
            "PR_AUC",
            "ROC_AUC",
            "Lift_10",
            "Lift_20",
            "Brier_Score",
        ]
    )
)

In [ ]:
rf_pr_auc_wins = (
    (
        temporal_pivot[
            "PR_AUC"
        ][
            "Random Forest"
        ]
        >
        temporal_pivot[
            "PR_AUC"
        ][
            "Reduced Logistic"
        ]
    )
    .sum()
)


total_temporal_comparisons = (
    len(
        temporal_pivot
    )
)


print(
    f"Random Forest PR-AUC wins: "
    f"{rf_pr_auc_wins} / "
    f"{total_temporal_comparisons}"
)

In [ ]:
rf_roc_wins = (
    (
        temporal_pivot[
            "ROC_AUC"
        ][
            "Random Forest"
        ]
        >
        temporal_pivot[
            "ROC_AUC"
        ][
            "Reduced Logistic"
        ]
    )
    .sum()
)


print(
    f"Random Forest ROC-AUC wins: "
    f"{rf_roc_wins} / "
    f"{total_temporal_comparisons}"
)

In [ ]:
rf_lift20_wins = (
    (
        temporal_pivot[
            "Lift_20"
        ][
            "Random Forest"
        ]
        >
        temporal_pivot[
            "Lift_20"
        ][
            "Reduced Logistic"
        ]
    )
    .sum()
)


print(
    f"Random Forest Lift@20 wins: "
    f"{rf_lift20_wins} / "
    f"{total_temporal_comparisons}"
)

In [ ]:
# ================================================================
# 35.8 — RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

display(
    random_forest_feature_importance[
        [
            "clean_feature",
            "importance"
        ]
    ]
    .head(25)
)

In [ ]:
# ================================================================
# 35.9 — CUMULATIVE FEATURE IMPORTANCE
# ================================================================

rf_importance_cumulative = (
    random_forest_feature_importance[
        [
            "clean_feature",
            "importance"
        ]
    ]
    .copy()
)


rf_importance_cumulative[
    "cumulative_importance"
] = (
    rf_importance_cumulative[
        "importance"
    ]
    .cumsum()
)


display(
    rf_importance_cumulative.head(25)
)

In [ ]:
# ================================================================
# 35.10 — HIGH-DRIFT FEATURE REGISTER
# ================================================================

high_drift_features = (
    reactivation_feature_drift.loc[
        reactivation_feature_drift[
            "absolute_SMD"
        ]
        >= 0.25,
        "feature"
    ]
    .tolist()
)


print(
    "High-drift features:"
)

print(
    high_drift_features
)

In [ ]:
# ================================================================
# 35.11 — FEATURE IMPORTANCE VS DRIFT
# ================================================================

rf_importance_vs_drift = (
    random_forest_feature_importance[
        [
            "clean_feature",
            "importance"
        ]
    ]
    .merge(
        reactivation_feature_drift[
            [
                "feature",
                "absolute_SMD"
            ]
        ],
        left_on="clean_feature",
        right_on="feature",
        how="left"
    )
)


rf_importance_vs_drift[
    "high_drift"
] = (
    rf_importance_vs_drift[
        "absolute_SMD"
    ]
    >= 0.25
)


display(
    rf_importance_vs_drift
    .sort_values(
        "importance",
        ascending=False
    )
    .head(25)
)

In [ ]:
# ================================================================
# 35.12 — IMPORTANCE DEPENDENCE ON HIGH-DRIFT VARIABLES
# ================================================================

known_drift_importance = (
    rf_importance_vs_drift.loc[
        rf_importance_vs_drift[
            "absolute_SMD"
        ]
        .notna()
    ]
)


high_drift_importance_share = (
    known_drift_importance.loc[
        known_drift_importance[
            "high_drift"
        ],
        "importance"
    ]
    .sum()
    /
    known_drift_importance[
        "importance"
    ]
    .sum()
)


print(
    "Share of measurable Random Forest feature importance "
    "coming from high-drift variables: "
    f"{high_drift_importance_share * 100:.2f}%"
)

In [ ]:
# ================================================================
# 35.13 — LOW-DRIFT RANDOM FOREST FEATURE SET
# ================================================================

low_drift_numeric_features = (
    reactivation_feature_drift.loc[
        reactivation_feature_drift[
            "absolute_SMD"
        ]
        <
        0.25,
        "feature"
    ]
    .tolist()
)


low_drift_categorical_features = [
    "last_order_region"
]


low_drift_reactivation_features = (
    low_drift_numeric_features
    +
    low_drift_categorical_features
)


print(
    f"Low-drift numeric features: "
    f"{len(low_drift_numeric_features)}"
)

print(
    low_drift_numeric_features
)

In [ ]:
# ================================================================
# 35.14 — LOW-DRIFT TREE PREPROCESSOR
# ================================================================

low_drift_tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True
                        )
                    )
                ]
            ),
            low_drift_numeric_features,
        ),

        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    ),
                ]
            ),
            low_drift_categorical_features,
        ),
    ]
)

### 35.15 — Train Low-Drift Random Forest

In [ ]:
# ================================================================
# 35.15 — LOW-DRIFT RANDOM FOREST
# ================================================================

low_drift_random_forest_model = Pipeline(
    steps=[
        (
            "preprocessing",
            low_drift_tree_preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=15,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            )
        ),
    ]
)


low_drift_random_forest_model.fit(
    X_reactivation_train[
        low_drift_reactivation_features
    ],
    y_reactivation_train
)

In [ ]:
validation_low_drift_rf_probabilities = (
    low_drift_random_forest_model
    .predict_proba(
        X_reactivation_validation[
            low_drift_reactivation_features
        ]
    )[:, 1]
)

In [ ]:
low_drift_rf_results = (
    evaluate_reactivation_model(
        model_name="Low-Drift Random Forest",
        y_true=y_reactivation_validation,
        probabilities=validation_low_drift_rf_probabilities,
    )
)


display(
    low_drift_rf_results
)

In [ ]:
# ================================================================
# 35.16 — RANDOM FOREST STRESS TEST
# ================================================================

rf_stress_test_comparison = (
    pd.concat(
        [
            random_forest_reactivation_results,
            low_drift_rf_results,
            reduced_reactivation_logistic_results,
        ],
        ignore_index=True
    )
)


display(
    rf_stress_test_comparison
)

### 35.17 — Calibration curves

In [ ]:
# ================================================================
# 35.17A — CALIBRATION IMPORT
# ================================================================

from sklearn.calibration import calibration_curve

print("✓ calibration_curve imported successfully.")

In [ ]:
# ================================================================
# 35.17B — CALIBRATION CANDIDATES
# ================================================================

calibration_candidates = [
    (
        "Random Forest",
        validation_random_forest_probabilities
    ),
    (
        "Reduced Logistic",
        validation_reduced_reactivation_probabilities
    ),
    (
        "Gradient Boosting",
        validation_gradient_boosting_probabilities
    ),
]


for model_name, probabilities in calibration_candidates:

    print(
        model_name,
        len(probabilities)
    )

In [ ]:
# ================================================================
# 35.17C — VALIDATION CALIBRATION CURVES
# ================================================================

plt.figure(
    figsize=(9, 7)
)


for model_name, probabilities in calibration_candidates:

    prob_true, prob_pred = calibration_curve(
        y_true=y_reactivation_validation,
        y_prob=probabilities,
        n_bins=10,
        strategy="quantile"
    )


    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label=model_name
    )


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)


plt.title(
    "Reactivation Model Calibration — Validation Population"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Reactivation Rate"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
# ================================================================
# 35.18 — MEAN ABSOLUTE CALIBRATION ERROR
# ================================================================

def mean_absolute_calibration_error(
    y_true,
    probabilities,
    n_bins=10
):

    prob_true, prob_pred = calibration_curve(
        y_true=y_true,
        y_prob=probabilities,
        n_bins=n_bins,
        strategy="quantile"
    )


    calibration_error = np.mean(
        np.abs(
            prob_pred
            -
            prob_true
        )
    )


    return (
        calibration_error
        *
        100
    )

In [ ]:
reactivation_calibration_summary = pd.DataFrame(
    {
        "Model": [
            "Random Forest",
            "Reduced Logistic",
            "Gradient Boosting",
            "HistGradientBoosting",
        ],

        "Mean Absolute Calibration Error (pp)": [
            mean_absolute_calibration_error(
                y_reactivation_validation,
                validation_random_forest_probabilities
            ),

            mean_absolute_calibration_error(
                y_reactivation_validation,
                validation_reduced_reactivation_probabilities
            ),

            mean_absolute_calibration_error(
                y_reactivation_validation,
                validation_gradient_boosting_probabilities
            ),

            mean_absolute_calibration_error(
                y_reactivation_validation,
                validation_hist_gradient_probabilities
            ),
        ],
    }
)


display(
    reactivation_calibration_summary
)

In [ ]:
# ================================================================
# 35.18B — CALIBRATION VALIDATION
# ================================================================

assert len(
    validation_random_forest_probabilities
) == len(
    y_reactivation_validation
)


assert len(
    validation_reduced_reactivation_probabilities
) == len(
    y_reactivation_validation
)


assert len(
    validation_gradient_boosting_probabilities
) == len(
    y_reactivation_validation
)


assert len(
    validation_hist_gradient_probabilities
) == len(
    y_reactivation_validation
)


assert (
    reactivation_calibration_summary[
        "Mean Absolute Calibration Error (pp)"
    ]
    .notna()
    .all()
)


print(
    "✓ Calibration analysis successfully completed."
)

In [ ]:
# ================================================================
# 35.19 — PAIRED BOOTSTRAP MODEL COMPARISON
# ================================================================

def paired_model_bootstrap(
    y_true,
    probabilities_a,
    probabilities_b,
    n_bootstrap=2000,
    random_state=42
):

    y_true = np.asarray(
        y_true
    )

    probabilities_a = np.asarray(
        probabilities_a
    )

    probabilities_b = np.asarray(
        probabilities_b
    )


    rng = np.random.default_rng(
        random_state
    )


    n = len(
        y_true
    )


    records = []


    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )


        y_sample = (
            y_true[
                indices
            ]
        )


        if (
            np.unique(
                y_sample
            ).size
            < 2
        ):
            continue


        a_sample = (
            probabilities_a[
                indices
            ]
        )


        b_sample = (
            probabilities_b[
                indices
            ]
        )


        records.append(
            {
                "PR_AUC_A":
                    average_precision_score(
                        y_sample,
                        a_sample
                    ),

                "PR_AUC_B":
                    average_precision_score(
                        y_sample,
                        b_sample
                    ),

                "ROC_AUC_A":
                    roc_auc_score(
                        y_sample,
                        a_sample
                    ),

                "ROC_AUC_B":
                    roc_auc_score(
                        y_sample,
                        b_sample
                    ),

                "Lift20_A":
                    lift_at_fraction(
                        y_sample,
                        a_sample,
                        0.20
                    ),

                "Lift20_B":
                    lift_at_fraction(
                        y_sample,
                        b_sample,
                        0.20
                    ),
            }
        )


    bootstrap = pd.DataFrame(
        records
    )


    bootstrap[
        "PR_AUC_Difference"
    ] = (
        bootstrap[
            "PR_AUC_A"
        ]
        -
        bootstrap[
            "PR_AUC_B"
        ]
    )


    bootstrap[
        "ROC_AUC_Difference"
    ] = (
        bootstrap[
            "ROC_AUC_A"
        ]
        -
        bootstrap[
            "ROC_AUC_B"
        ]
    )


    bootstrap[
        "Lift20_Difference"
    ] = (
        bootstrap[
            "Lift20_A"
        ]
        -
        bootstrap[
            "Lift20_B"
        ]
    )


    return bootstrap

In [ ]:
rf_vs_logistic_bootstrap = (
    paired_model_bootstrap(
        y_true=y_reactivation_validation,
        probabilities_a=validation_random_forest_probabilities,
        probabilities_b=validation_reduced_reactivation_probabilities,
        n_bootstrap=2000,
        random_state=42
    )
)

In [ ]:
# ================================================================
# 35.20 — PAIRED BOOTSTRAP SUMMARY
# ================================================================

bootstrap_difference_summary = pd.DataFrame(
    {
        "Metric Difference": [
            "RF - Reduced Logistic PR-AUC",
            "RF - Reduced Logistic ROC-AUC",
            "RF - Reduced Logistic Lift@20",
        ],

        "Mean Difference": [
            rf_vs_logistic_bootstrap[
                "PR_AUC_Difference"
            ].mean(),

            rf_vs_logistic_bootstrap[
                "ROC_AUC_Difference"
            ].mean(),

            rf_vs_logistic_bootstrap[
                "Lift20_Difference"
            ].mean(),
        ],

        "CI 2.5%": [
            rf_vs_logistic_bootstrap[
                "PR_AUC_Difference"
            ].quantile(
                0.025
            ),

            rf_vs_logistic_bootstrap[
                "ROC_AUC_Difference"
            ].quantile(
                0.025
            ),

            rf_vs_logistic_bootstrap[
                "Lift20_Difference"
            ].quantile(
                0.025
            ),
        ],

        "CI 97.5%": [
            rf_vs_logistic_bootstrap[
                "PR_AUC_Difference"
            ].quantile(
                0.975
            ),

            rf_vs_logistic_bootstrap[
                "ROC_AUC_Difference"
            ].quantile(
                0.975
            ),

            rf_vs_logistic_bootstrap[
                "Lift20_Difference"
            ].quantile(
                0.975
            ),
        ],

        "Probability RF Better": [
            (
                rf_vs_logistic_bootstrap[
                    "PR_AUC_Difference"
                ]
                > 0
            ).mean(),

            (
                rf_vs_logistic_bootstrap[
                    "ROC_AUC_Difference"
                ]
                > 0
            ).mean(),

            (
                rf_vs_logistic_bootstrap[
                    "Lift20_Difference"
                ]
                > 0
            ).mean(),
        ],
    }
)


display(
    bootstrap_difference_summary
)

In [ ]:
# ================================================================
# 35.21 — RANDOM FOREST DECILES
# ================================================================

random_forest_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_random_forest_probabilities,
        n_deciles=10
    )
)


display(
    random_forest_reactivation_deciles
)

In [ ]:
# ================================================================
# 35.22 — REDUCED LOGISTIC DECILES
# ================================================================

reduced_reactivation_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_validation,
        probabilities=validation_reduced_reactivation_probabilities,
        n_deciles=10
    )
)


display(
    reduced_reactivation_deciles
)

In [ ]:
rf_deciles_plot = (
    random_forest_reactivation_deciles[
        [
            "score_decile",
            "actual_reactivation_rate_pct"
        ]
    ]
    .copy()
)


rf_deciles_plot[
    "Model"
] = "Random Forest"


logistic_deciles_plot = (
    reduced_reactivation_deciles[
        [
            "score_decile",
            "actual_reactivation_rate_pct"
        ]
    ]
    .copy()
)


logistic_deciles_plot[
    "Model"
] = "Reduced Logistic"


champion_decile_plot = pd.concat(
    [
        rf_deciles_plot,
        logistic_deciles_plot
    ],
    ignore_index=True
)


plt.figure(
    figsize=(11, 6)
)


sns.lineplot(
    data=champion_decile_plot,
    x="score_decile",
    y="actual_reactivation_rate_pct",
    hue="Model",
    marker="o"
)


plt.axhline(
    y=validation_reactivation_prevalence * 100,
    linestyle="--"
)


plt.title(
    "Validation Reactivation Rate by Model Score Decile"
)

plt.xlabel(
    "Score Decile — 1 = Highest Reactivation Propensity"
)

plt.ylabel(
    "Observed Reactivation Rate (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 35.23 — FINAL PRE-TEST SCORECARD
# ================================================================

pretest_reactivation_scorecard = pd.DataFrame(
    [
        {
            "Model":
                "Random Forest",

            "PR-AUC":
                random_forest_reactivation_results[
                    "PR-AUC"
                ].iloc[0],

            "ROC-AUC":
                random_forest_reactivation_results[
                    "ROC-AUC"
                ].iloc[0],

            "Brier Score":
                random_forest_reactivation_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                random_forest_reactivation_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                random_forest_reactivation_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Complexity":
                "Higher",

            "Interpretability":
                "Moderate / feature importance"
        },

        {
            "Model":
                "Reduced Logistic",

            "PR-AUC":
                reduced_reactivation_logistic_results[
                    "PR-AUC"
                ].iloc[0],

            "ROC-AUC":
                reduced_reactivation_logistic_results[
                    "ROC-AUC"
                ].iloc[0],

            "Brier Score":
                reduced_reactivation_logistic_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                reduced_reactivation_logistic_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                reduced_reactivation_logistic_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Complexity":
                "Lower",

            "Interpretability":
                "High"
        },
    ]
)


display(
    pretest_reactivation_scorecard
)

In [ ]:
REACTIVATION_MODEL_FROZEN = False

print(
    "Model remains unfrozen pending Step 35 diagnostics."
)

In [ ]:
# ================================================================
# 35.25 — STEP 35 CHECKS
# ================================================================

assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


assert (
    len(
        temporal_reactivation_results
    )
    > 0
)


assert (
    len(
        rf_vs_logistic_bootstrap
    )
    > 0
)


assert (
    REACTIVATION_MODEL_FROZEN
    is False
)


print(
    "✓ Step 35 diagnostics complete."
)

print(
    "✓ Temporal stability evaluated."
)

print(
    "✓ Random Forest dependence on drifting features assessed."
)

print(
    "✓ Calibration assessed."
)

print(
    "✓ Paired validation bootstrap completed."
)

print(
    "✓ Reactivation Test population remains unopened."
)

### Step 35 Summary — Temporal Stability, Feature Dependence and Calibration

Step 35 tested whether the Random Forest advantage identified during the primary validation period was stable enough to justify freezing it as the final reactivation model.

The final Test population remained unopened.

### Random Forest Did Not Consistently Dominate Across Historical Periods

Across eight expanding chronological validation folds, Random Forest and Reduced Logistic Regression produced broadly similar performance.

Average historical performance was approximately:

| Metric | Random Forest | Reduced Logistic |
|---|---:|---:|
| PR-AUC | 0.494 | 0.502 |
| ROC-AUC | 0.660 | 0.667 |
| Lift@10% | 1.76× | 1.77× |
| Lift@20% | 1.62× | 1.63× |

Random Forest achieved better PR-AUC in only **3 of 8 folds**, better ROC-AUC in **4 of 8 folds**, and better Lift@20% in **4 of 8 folds**.

The Random Forest advantage observed in the principal validation population therefore does not appear to be uniformly persistent through historical time.

### Random Forest Remains Strong on the Main Validation Population

Despite mixed historical fold results, Random Forest remains the strongest model on the designated later Validation population.

It achieved approximately:

- **PR-AUC: 0.607**
- **ROC-AUC: 0.721**
- **Lift@10%: 2.00×**
- **Lift@20%: 1.81×**
- **Brier score: 0.203**

These remain the strongest overall metrics observed in the primary validation exercise.

### Bootstrap Analysis Shows Probable but Not Conclusive Superiority

Paired bootstrap analysis compared Random Forest directly with Reduced Logistic Regression on the same validation observations.

The estimated Random Forest advantage was approximately:

- **PR-AUC: +0.024**
- **ROC-AUC: +0.013**
- **Lift@20%: +0.114**

Random Forest outperformed Reduced Logistic in approximately **88%–90% of bootstrap resamples**, depending on the metric.

However, the 95% confidence intervals for all three performance differences crossed zero.

The evidence therefore suggests that Random Forest is probably superior on the designated validation population, but the magnitude of its advantage remains uncertain.

### A Low-Drift Random Forest Preserved Almost All Predictive Performance

A Random Forest challenger was trained after removing numeric features with absolute Train-to-Validation SMD of 0.25 or greater.

The Low-Drift Random Forest achieved approximately:

- **PR-AUC: 0.604**
- **ROC-AUC: 0.724**
- **Brier score: 0.205**
- **Lift@10%: 1.97×**
- **Lift@20%: 1.79×**

The PR-AUC difference from the Full Random Forest was only approximately **0.003**, which falls inside the predetermined practical-equivalence tolerance of 0.005.

The Low-Drift Random Forest also slightly exceeded the Full Random Forest on ROC-AUC.

This is an important result because it demonstrates that most of the Random Forest's predictive performance can be retained without relying on the most temporally unstable numeric features.

### Random Forest Is Not Entirely Driven by High-Drift Variables

Approximately **33.2%** of measurable Random Forest feature importance originated from features classified as high drift.

The majority of measurable importance therefore originated from comparatively more stable variables.

This reduces concern that the model's validation performance is entirely dependent on temporal maturity effects.

### Calendar Timing Is an Important Predictor

Prediction month was the single largest Random Forest feature importance.

This indicates that reactivation behaviour contains meaningful calendar or cohort structure.

Although calendar information is legitimately available at prediction time, reliance on calendar timing introduces a generalisation risk because seasonal or operating conditions may change in future periods.

This further strengthens the case for comparing the Full Random Forest with its Low-Drift variant before final model freeze.

### Reduced Logistic Remains Better Calibrated

Mean absolute calibration error was approximately:

- **Reduced Logistic: 4.63 percentage points**
- **Random Forest: 5.87 percentage points**
- **HistGradientBoosting: 9.77 percentage points**
- **Gradient Boosting: 9.90 percentage points**

Reduced Logistic therefore provides somewhat better probability calibration.

Random Forest remains stronger for ranking on the designated validation population, while Reduced Logistic remains attractive when simplicity and probability interpretability are prioritised.

### Step 35 Decision

No model is frozen yet.

Three finalists remain:

1. **Full Random Forest**
2. **Low-Drift Random Forest**
3. **Reduced Logistic Regression**

The Low-Drift Random Forest is particularly important because it retains essentially all of the Full Random Forest's validation discrimination while reducing dependence on features that change materially through time.

The next step will subject the Low-Drift Random Forest to the same historical temporal-stability diagnostics before one final model specification is frozen.

The Test population remains unopened.



## Step 36 — Final Reactivation Model Selection and Freeze

Step 36 performs the final pre-test comparison of the three remaining reactivation-model candidates:

1. Full Random Forest;
2. Low-Drift Random Forest;
3. Reduced Logistic Regression.

The Low-Drift Random Forest retained almost all of the Full Random Forest's designated validation performance while excluding the most temporally unstable numeric features.

Before freezing the model, this step evaluates whether that performance also remains credible across historical temporal folds.

No additional model families will be introduced.

After the final specification is frozen:

- no features will be added or removed;
- no algorithm will be changed;
- no hyperparameters will be altered;
- no calibration decision will be changed based on Test performance;
- and the Test population will be opened exactly once for final out-of-time evaluation.

In [ ]:
# ================================================================
# 36.2 — PRE-FREEZE GOVERNANCE
# ================================================================

assert REACTIVATION_TEST_SET_OPENED is False
assert REACTIVATION_MODEL_FROZEN is False

print("✓ Test remains unopened.")
print("✓ Final model has not yet been frozen.")

In [ ]:
# ================================================================
# 36.3 — LOW-DRIFT RANDOM FOREST TEMPORAL VALIDATION
# ================================================================

low_drift_temporal_records = []


for fold_number, fold in enumerate(
    reactivation_temporal_folds,
    start=1
):

    fold_train = (
        reactivation_train_ordered.loc[
            fold["train_indices"]
        ]
    )

    fold_validation = (
        reactivation_train_ordered.loc[
            fold["validation_indices"]
        ]
    )


    y_fold_train = (
        fold_train[
            reactivation_target
        ]
    )


    y_fold_validation = (
        fold_validation[
            reactivation_target
        ]
    )


    if y_fold_validation.nunique() < 2:
        continue


    fold_model = clone(
        low_drift_random_forest_model
    )


    fold_model.fit(
        fold_train[
            low_drift_reactivation_features
        ],
        y_fold_train
    )


    fold_probabilities = (
        fold_model
        .predict_proba(
            fold_validation[
                low_drift_reactivation_features
            ]
        )[:, 1]
    )


    low_drift_temporal_records.append(
        {
            "fold":
                fold_number,

            "validation_period":
                str(
                    fold[
                        "validation_period"
                    ]
                ),

            "model":
                "Low-Drift Random Forest",

            "train_episodes":
                len(
                    fold_train
                ),

            "validation_episodes":
                len(
                    fold_validation
                ),

            "validation_prevalence":
                y_fold_validation.mean(),

            "PR_AUC":
                average_precision_score(
                    y_fold_validation,
                    fold_probabilities
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_fold_validation,
                    fold_probabilities
                ),

            "Brier_Score":
                brier_score_loss(
                    y_fold_validation,
                    fold_probabilities
                ),

            "Lift_10":
                lift_at_fraction(
                    y_fold_validation,
                    fold_probabilities,
                    0.10
                ),

            "Lift_20":
                lift_at_fraction(
                    y_fold_validation,
                    fold_probabilities,
                    0.20
                ),
        }
    )


low_drift_temporal_results = pd.DataFrame(
    low_drift_temporal_records
)


display(
    low_drift_temporal_results
)

In [ ]:
# ================================================================
# 36.4 — THREE-MODEL TEMPORAL COMPARISON
# ================================================================

three_model_temporal_results = pd.concat(
    [
        temporal_reactivation_results,
        low_drift_temporal_results,
    ],
    ignore_index=True
)


three_model_temporal_summary = (
    three_model_temporal_results
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        folds=("fold", "nunique"),

        mean_PR_AUC=("PR_AUC", "mean"),
        std_PR_AUC=("PR_AUC", "std"),
        min_PR_AUC=("PR_AUC", "min"),

        mean_ROC_AUC=("ROC_AUC", "mean"),
        std_ROC_AUC=("ROC_AUC", "std"),

        mean_Lift_10=("Lift_10", "mean"),
        min_Lift_10=("Lift_10", "min"),

        mean_Lift_20=("Lift_20", "mean"),
        min_Lift_20=("Lift_20", "min"),

        mean_Brier=("Brier_Score", "mean"),
    )
)


display(
    three_model_temporal_summary
)

In [ ]:
# ================================================================
# 36.5 — LOW-DRIFT RF CALIBRATION
# ================================================================

low_drift_rf_calibration_error = (
    mean_absolute_calibration_error(
        y_reactivation_validation,
        validation_low_drift_rf_probabilities
    )
)


print(
    "Low-Drift Random Forest calibration error: "
    f"{low_drift_rf_calibration_error:.2f} pp"
)

In [ ]:
# ================================================================
# 36.6 — FINAL VALIDATION SCORECARD
# ================================================================

final_reactivation_scorecard = pd.DataFrame(
    [
        {
            "Model":
                "Full Random Forest",

            "PR-AUC":
                random_forest_reactivation_results[
                    "PR-AUC"
                ].iloc[0],

            "ROC-AUC":
                random_forest_reactivation_results[
                    "ROC-AUC"
                ].iloc[0],

            "Brier":
                random_forest_reactivation_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                random_forest_reactivation_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                random_forest_reactivation_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Calibration Error pp":
                reactivation_calibration_summary.loc[
                    reactivation_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Random Forest",
                    "Mean Absolute Calibration Error (pp)"
                ].iloc[0],

            "Feature Set":
                "Full"
        },

        {
            "Model":
                "Low-Drift Random Forest",

            "PR-AUC":
                low_drift_rf_results[
                    "PR-AUC"
                ].iloc[0],

            "ROC-AUC":
                low_drift_rf_results[
                    "ROC-AUC"
                ].iloc[0],

            "Brier":
                low_drift_rf_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                low_drift_rf_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                low_drift_rf_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Calibration Error pp":
                low_drift_rf_calibration_error,

            "Feature Set":
                "Low-drift"
        },

        {
            "Model":
                "Reduced Logistic",

            "PR-AUC":
                reduced_reactivation_logistic_results[
                    "PR-AUC"
                ].iloc[0],

            "ROC-AUC":
                reduced_reactivation_logistic_results[
                    "ROC-AUC"
                ].iloc[0],

            "Brier":
                reduced_reactivation_logistic_results[
                    "Brier Score"
                ].iloc[0],

            "Lift@10":
                reduced_reactivation_logistic_results[
                    "Lift @ 10%"
                ].iloc[0],

            "Lift@20":
                reduced_reactivation_logistic_results[
                    "Lift @ 20%"
                ].iloc[0],

            "Calibration Error pp":
                reactivation_calibration_summary.loc[
                    reactivation_calibration_summary[
                        "Model"
                    ]
                    ==
                    "Reduced Logistic",
                    "Mean Absolute Calibration Error (pp)"
                ].iloc[0],

            "Feature Set":
                "Reduced / interpretable"
        },
    ]
)


display(
    final_reactivation_scorecard
)

In [ ]:
# ================================================================
# 36.7 — FINAL PRACTICAL EQUIVALENCE
# ================================================================

FINAL_PR_AUC_TOLERANCE = 0.005


best_final_pr_auc = (
    final_reactivation_scorecard[
        "PR-AUC"
    ]
    .max()
)


final_reactivation_scorecard[
    "Within_PR_AUC_Equivalence_Band"
] = (
    final_reactivation_scorecard[
        "PR-AUC"
    ]
    >=
    (
        best_final_pr_auc
        -
        FINAL_PR_AUC_TOLERANCE
    )
)


display(
    final_reactivation_scorecard
)

### Final Model Selection Rule

The final model will be selected using the following hierarchy:

1. Models outside the predetermined PR-AUC practical-equivalence band are removed.
2. Among equivalent models, temporal stability is preferred over isolated validation advantage.
3. If temporal performance remains comparable, lower dependence on highly drifting variables is preferred.
4. If still comparable, better calibration is preferred.
5. If still comparable, lower complexity is preferred.

The purpose is to select the model most likely to generalise rather than the model with the single highest validation statistic.

In [ ]:
# ================================================================
# 36.9 — TEMPORAL RANK SCORECARD
# ================================================================

temporal_rank_scorecard = (
    three_model_temporal_summary[
        [
            "model",
            "mean_PR_AUC",
            "std_PR_AUC",
            "mean_ROC_AUC",
            "mean_Lift_10",
            "mean_Lift_20",
            "min_Lift_20",
            "mean_Brier",
        ]
    ]
    .copy()
)


display(
    temporal_rank_scorecard
)

### 36.10 — Do not automate final selection yet
I deliberately don't want code to blindly choose the champion.

The final decision should combine the evidence from:
- 36.6;
- 36.7;
- 36.9.
So leave:

In [ ]:
# ================================================================
# 36.10 — FINAL SELECTION PENDING EVIDENCE REVIEW
# ================================================================

FINAL_REACTIVATION_MODEL_NAME = None
FINAL_REACTIVATION_MODEL = None
FINAL_REACTIVATION_FEATURES = None


print(
    "Final model selection pending Step 36 evidence review."
)

### Step 36 Summary — Final Reactivation Model Selection and Freeze

Step 36 completed the final pre-test model-selection process.

Three candidate models remained:

1. Full Random Forest;
2. Low-Drift Random Forest;
3. Reduced Logistic Regression.

The selection process followed a predetermined hierarchy based on validation discrimination, practical equivalence, temporal stability, feature drift, calibration and model complexity.

### Reduced Logistic Did Not Qualify for the Final Equivalence Band

Reduced Logistic Regression remained an attractive model because of its simplicity, interpretability and calibration.

Its designated validation performance was approximately:

- **PR-AUC: 0.584**
- **ROC-AUC: 0.708**
- **Lift@10%: 1.77×**
- **Lift@20%: 1.71×**
- **Calibration error: 4.63 percentage points**

It also produced slightly stronger average performance across several historical temporal folds.

However, its validation PR-AUC was approximately 0.023 below the Full Random Forest.

This difference exceeds the predetermined practical-equivalence tolerance of 0.005.

Reduced Logistic was therefore removed from final model contention despite remaining an important interpretable benchmark.

### Full and Low-Drift Random Forest Were Practically Equivalent on Validation PR-AUC

The Full Random Forest achieved validation PR-AUC of approximately **0.6071**.

The Low-Drift Random Forest achieved approximately **0.6044**.

The difference of approximately **0.0027** falls inside the predetermined PR-AUC equivalence band.

The two Random Forest specifications were therefore compared using secondary evidence.

### Full Random Forest Retained the Stronger Overall Profile

The Full Random Forest produced approximately:

- **ROC-AUC: 0.721**
- **Brier score: 0.203**
- **Lift@10%: 2.00×**
- **Lift@20%: 1.81×**
- **Calibration error: 5.87 percentage points**

The Low-Drift Random Forest produced approximately:

- **ROC-AUC: 0.724**
- **Brier score: 0.205**
- **Lift@10%: 1.97×**
- **Lift@20%: 1.79×**
- **Calibration error: 6.90 percentage points**

Although the Low-Drift model achieved slightly higher ROC-AUC, the Full Random Forest retained stronger PR-AUC, Brier score, top-of-ranking lift and calibration.

### Historical Temporal Performance Was Comparable

Across eight expanding historical folds, the Full Random Forest achieved approximately:

- **mean PR-AUC: 0.494**
- **mean ROC-AUC: 0.660**
- **mean Lift@10%: 1.76×**
- **mean Lift@20%: 1.62×**
- **minimum Lift@20%: 1.14×**

The Low-Drift Random Forest achieved approximately:

- **mean PR-AUC: 0.491**
- **mean ROC-AUC: 0.657**
- **mean Lift@10%: 1.65×**
- **mean Lift@20%: 1.59×**
- **minimum Lift@20%: 1.11×**

The Full Random Forest therefore retained slightly stronger historical average and commercial-ranking performance.

### Dependence on High-Drift Features Was Material but Not Dominant

Approximately **33.2%** of measurable Random Forest feature importance originated from variables classified as high drift.

The majority of measurable model importance therefore came from comparatively more stable variables.

The ability of the Low-Drift Random Forest to retain similar validation performance provides useful evidence that the model's signal is not entirely dependent on unstable customer-maturity features.

However, the Full Random Forest still produced the strongest overall validation and historical evidence.

### Final Pre-Test Model Decision

The final frozen reactivation model is:

> **Full Random Forest**

The frozen specification is:

- **algorithm:** Random Forest Classifier;
- **estimators:** 500;
- **maximum tree depth:** 8;
- **minimum samples per leaf:** 15;
- **maximum features per split:** square root;
- **numeric preprocessing:** median imputation with missingness indicators;
- **categorical preprocessing:** most-frequent imputation followed by one-hot encoding;
- **primary selection metric:** PR-AUC;
- **commercial ranking metrics:** Lift@10% and Lift@20%;
- **validation design:** chronological;
- **calibration layer:** none.

No further feature selection, model-family comparison, hyperparameter tuning or validation-driven model modification is permitted after this point.

The untouched Test population can now be opened exactly once for final temporal evaluation.

In [ ]:
# ================================================================
# 36.11 — FREEZE FINAL REACTIVATION MODEL
# ================================================================

FINAL_REACTIVATION_MODEL_NAME = (
    "Full Random Forest"
)


FINAL_REACTIVATION_MODEL = (
    random_forest_reactivation_model
)


FINAL_REACTIVATION_FEATURES = (
    reactivation_model_features.copy()
)


REACTIVATION_MODEL_FROZEN = True


print(
    f"Frozen reactivation model: "
    f"{FINAL_REACTIVATION_MODEL_NAME}"
)

print(
    f"Frozen feature count: "
    f"{len(FINAL_REACTIVATION_FEATURES)}"
)

print(
    "✓ Reactivation model specification is now frozen."
)

In [ ]:
# ================================================================
# 36.12 — FROZEN MODEL SPECIFICATION
# ================================================================

frozen_reactivation_specification = pd.DataFrame(
    {
        "Component": [
            "Prediction problem",
            "Prediction point",
            "Target horizon",
            "Model",
            "Estimators",
            "Maximum depth",
            "Minimum leaf size",
            "Maximum features",
            "Numeric imputation",
            "Missingness indicators",
            "Categorical imputation",
            "Categorical encoding",
            "Calibration",
            "Primary selection metric",
            "Commercial metrics",
            "Validation design",
            "Test status",
        ],

        "Frozen Specification": [
            "90-day customer reactivation",
            "Exactly 90 days after previous qualifying purchase",
            "Next 90 days",
            "Random Forest Classifier",
            500,
            8,
            15,
            "sqrt",
            "Median",
            "Yes",
            "Most frequent",
            "One-hot encoding",
            "None",
            "PR-AUC",
            "Lift@10%, Lift@20%, Capture@10%, Capture@20%",
            "Chronological out-of-time",
            "UNOPENED",
        ],
    }
)


display(
    frozen_reactivation_specification
)

In [ ]:
# ================================================================
# 36.13 — FREEZE VALIDATION BENCHMARK
# ================================================================

frozen_reactivation_validation_benchmark = (
    random_forest_reactivation_results
    .copy()
)


frozen_reactivation_validation_benchmark[
    "Evaluation Population"
] = (
    "Validation"
)


display(
    frozen_reactivation_validation_benchmark
)

In [ ]:
# ================================================================
# 36.14 — GOVERNANCE LOCK
# ================================================================

assert (
    REACTIVATION_MODEL_FROZEN
    is True
)


assert (
    REACTIVATION_TEST_SET_OPENED
    is False
)


assert (
    FINAL_REACTIVATION_MODEL_NAME
    ==
    "Full Random Forest"
)


print(
    "✓ Model frozen."
)

print(
    "✓ Feature specification frozen."
)

print(
    "✓ Hyperparameters frozen."
)

print(
    "✓ Validation benchmark frozen."
)

print(
    "✓ Test population remains unopened."
)

print(
    "No further validation-driven model changes are permitted."
)

## Step 37 — Final Untouched Temporal Test Evaluation

Step 37 performs the first and only evaluation of the frozen reactivation model on the future Test population.

All model-development decisions have now been completed.

The frozen model is the **Full Random Forest** selected through chronological validation, model-family comparison, temporal stability testing, calibration assessment and feature-drift analysis.

Before Test scoring, the exact frozen model specification will be refitted using the combined Train + Validation development population.

No model architecture, feature, preprocessing rule or hyperparameter will be changed.

The future Test population will then be scored exactly once.

Test performance will not be used to modify the model.

The objective is to determine whether the reactivation signal identified during development genuinely generalises to later inactivity episodes.

In [ ]:
# ================================================================
# 37.2 — CONFIRM FREEZE
# ================================================================

assert REACTIVATION_MODEL_FROZEN is True
assert REACTIVATION_TEST_SET_OPENED is False

assert (
    FINAL_REACTIVATION_MODEL_NAME
    ==
    "Full Random Forest"
)


print(
    "✓ Frozen model confirmed."
)

print(
    "✓ Test remains unopened."
)

In [ ]:
# ================================================================
# 37.3 — TRAIN + VALIDATION DEVELOPMENT POPULATION
# ================================================================

reactivation_development_data = (
    pd.concat(
        [
            train_reactivation_data,
            validation_reactivation_data,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "prediction_date",
            "customer"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Train episodes: "
    f"{len(train_reactivation_data):,}"
)

print(
    f"Validation episodes: "
    f"{len(validation_reactivation_data):,}"
)

print(
    f"Combined development episodes: "
    f"{len(reactivation_development_data):,}"
)

In [ ]:
# ================================================================
# 37.4 — DEVELOPMENT MATRICES
# ================================================================

X_reactivation_development = (
    reactivation_development_data[
        FINAL_REACTIVATION_FEATURES
    ]
    .copy()
)


y_reactivation_development = (
    reactivation_development_data[
        reactivation_target
    ]
    .copy()
)


print(
    f"Development reactivation rate: "
    f"{y_reactivation_development.mean() * 100:.2f}%"
)

In [ ]:
# ================================================================
# 37.5 — CLONE FROZEN MODEL
# ================================================================

final_reactivation_refit_model = clone(
    FINAL_REACTIVATION_MODEL
)


print(
    "✓ Exact frozen Random Forest specification cloned."
)

In [ ]:
# ================================================================
# 37.6 — FINAL DEVELOPMENT REFIT
# ================================================================

final_reactivation_refit_model.fit(
    X_reactivation_development,
    y_reactivation_development
)


print(
    "✓ Frozen specification refitted on "
    "combined Train + Validation data."
)

In [ ]:
# ================================================================
# 37.7 — DEVELOPMENT / TEST TEMPORAL SEPARATION
# ================================================================

development_end = (
    reactivation_development_data[
        "prediction_date"
    ]
    .max()
)


test_start = (
    test_reactivation_data[
        "prediction_date"
    ]
    .min()
)


print(
    f"Development ends: "
    f"{development_end}"
)

print(
    f"Test begins: "
    f"{test_start}"
)


assert (
    development_end
    <
    test_start
)


print(
    "✓ Strict temporal separation confirmed."
)

In [ ]:
# ================================================================
# 37.8 — OPEN UNTOUCHED TEST POPULATION
# ================================================================

REACTIVATION_TEST_SET_OPENED = True


print(
    "FINAL REACTIVATION TEST POPULATION IS NOW OPEN."
)

print(
    "No further model changes are permitted."
)

In [ ]:
# ================================================================
# 37.9 — FINAL TEST PREDICTIONS
# ================================================================

reactivation_test_probabilities = (
    final_reactivation_refit_model
    .predict_proba(
        X_reactivation_test[
            FINAL_REACTIVATION_FEATURES
        ]
    )[:, 1]
)


assert (
    len(
        reactivation_test_probabilities
    )
    ==
    len(
        y_reactivation_test
    )
)


print(
    "✓ Final Test probabilities generated."
)

In [ ]:
# ================================================================
# 37.10 — FINAL REACTIVATION TEST METRICS
# ================================================================

final_reactivation_test_results = (
    evaluate_reactivation_model(
        model_name="Frozen Random Forest — Final Test",
        y_true=y_reactivation_test,
        probabilities=reactivation_test_probabilities,
    )
)


display(
    final_reactivation_test_results
)

In [ ]:
# ================================================================
# 37.11 — VALIDATION VS TEST
# ================================================================

reactivation_validation_vs_test = pd.DataFrame(
    {
        "Metric": [
            "PR-AUC",
            "ROC-AUC",
            "Brier Score",
            "Lift @ 10%",
            "Lift @ 20%",
            "Capture @ 10%",
            "Capture @ 20%",
        ],

        "Validation": [
            frozen_reactivation_validation_benchmark[
                "PR-AUC"
            ].iloc[0],

            frozen_reactivation_validation_benchmark[
                "ROC-AUC"
            ].iloc[0],

            frozen_reactivation_validation_benchmark[
                "Brier Score"
            ].iloc[0],

            frozen_reactivation_validation_benchmark[
                "Lift @ 10%"
            ].iloc[0],

            frozen_reactivation_validation_benchmark[
                "Lift @ 20%"
            ].iloc[0],

            frozen_reactivation_validation_benchmark[
                "Capture @ 10%"
            ].iloc[0],

            frozen_reactivation_validation_benchmark[
                "Capture @ 20%"
            ].iloc[0],
        ],

        "Test": [
            final_reactivation_test_results[
                "PR-AUC"
            ].iloc[0],

            final_reactivation_test_results[
                "ROC-AUC"
            ].iloc[0],

            final_reactivation_test_results[
                "Brier Score"
            ].iloc[0],

            final_reactivation_test_results[
                "Lift @ 10%"
            ].iloc[0],

            final_reactivation_test_results[
                "Lift @ 20%"
            ].iloc[0],

            final_reactivation_test_results[
                "Capture @ 10%"
            ].iloc[0],

            final_reactivation_test_results[
                "Capture @ 20%"
            ].iloc[0],
        ],
    }
)


reactivation_validation_vs_test[
    "Absolute Change"
] = (
    reactivation_validation_vs_test[
        "Test"
    ]
    -
    reactivation_validation_vs_test[
        "Validation"
    ]
)


display(
    reactivation_validation_vs_test
)

In [ ]:
# ================================================================
# 37.12 — PREVALENCE-AWARE PR-AUC
# ================================================================

validation_prevalence = (
    y_reactivation_validation.mean()
)


test_prevalence = (
    y_reactivation_test.mean()
)


validation_pr_auc = (
    frozen_reactivation_validation_benchmark[
        "PR-AUC"
    ].iloc[0]
)


test_pr_auc = (
    final_reactivation_test_results[
        "PR-AUC"
    ].iloc[0]
)


reactivation_prevalence_adjusted_pr_auc = pd.DataFrame(
    {
        "Dataset": [
            "Validation",
            "Test",
        ],

        "Prevalence": [
            validation_prevalence,
            test_prevalence,
        ],

        "PR-AUC": [
            validation_pr_auc,
            test_pr_auc,
        ],
    }
)

In [ ]:
reactivation_prevalence_adjusted_pr_auc[
    "PR-AUC / Prevalence"
] = (
    reactivation_prevalence_adjusted_pr_auc[
        "PR-AUC"
    ]
    /
    reactivation_prevalence_adjusted_pr_auc[
        "Prevalence"
    ]
)


reactivation_prevalence_adjusted_pr_auc[
    "Normalised PR-AUC"
] = (
    (
        reactivation_prevalence_adjusted_pr_auc[
            "PR-AUC"
        ]
        -
        reactivation_prevalence_adjusted_pr_auc[
            "Prevalence"
        ]
    )
    /
    (
        1
        -
        reactivation_prevalence_adjusted_pr_auc[
            "Prevalence"
        ]
    )
)


display(
    reactivation_prevalence_adjusted_pr_auc
)

In [ ]:
# ================================================================
# 37.13 — FINAL TEST DECILES
# ================================================================

final_reactivation_test_deciles = (
    create_reactivation_decile_performance(
        y_true=y_reactivation_test,
        probabilities=reactivation_test_probabilities,
        n_deciles=10
    )
)


display(
    final_reactivation_test_deciles
)

In [ ]:
# ================================================================
# 37.14 — FINAL TEST CALIBRATION
# ================================================================

final_test_calibration_error = (
    mean_absolute_calibration_error(
        y_reactivation_test,
        reactivation_test_probabilities
    )
)


print(
    "Final Test mean absolute calibration error: "
    f"{final_test_calibration_error:.2f} pp"
)

In [ ]:
# ================================================================
# 37.15 — FINAL TEST BOOTSTRAP
# ================================================================

def bootstrap_reactivation_metrics(
    y_true,
    probabilities,
    n_bootstrap=2000,
    random_state=42
):

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )


    rng = np.random.default_rng(
        random_state
    )


    n = len(
        y_true
    )


    records = []


    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )


        y_sample = y_true[
            indices
        ]


        probability_sample = probabilities[
            indices
        ]


        if np.unique(
            y_sample
        ).size < 2:

            continue


        records.append(
            {
                "PR-AUC":
                    average_precision_score(
                        y_sample,
                        probability_sample
                    ),

                "ROC-AUC":
                    roc_auc_score(
                        y_sample,
                        probability_sample
                    ),

                "Brier Score":
                    brier_score_loss(
                        y_sample,
                        probability_sample
                    ),

                "Lift @ 10%":
                    lift_at_fraction(
                        y_sample,
                        probability_sample,
                        0.10
                    ),

                "Lift @ 20%":
                    lift_at_fraction(
                        y_sample,
                        probability_sample,
                        0.20
                    ),
            }
        )


    return pd.DataFrame(
        records
    )

In [ ]:
reactivation_test_bootstrap = (
    bootstrap_reactivation_metrics(
        y_true=y_reactivation_test,
        probabilities=reactivation_test_probabilities,
        n_bootstrap=2000,
        random_state=42
    )
)

In [ ]:
# ================================================================
# 37.16 — FINAL TEST CONFIDENCE INTERVALS
# ================================================================

reactivation_test_bootstrap_summary = []


for metric in [
    "PR-AUC",
    "ROC-AUC",
    "Brier Score",
    "Lift @ 10%",
    "Lift @ 20%",
]:

    reactivation_test_bootstrap_summary.append(
        {
            "Metric":
                metric,

            "Estimate":
                final_reactivation_test_results[
                    metric
                ].iloc[0],

            "Bootstrap Mean":
                reactivation_test_bootstrap[
                    metric
                ].mean(),

            "CI 2.5%":
                reactivation_test_bootstrap[
                    metric
                ].quantile(
                    0.025
                ),

            "CI 97.5%":
                reactivation_test_bootstrap[
                    metric
                ].quantile(
                    0.975
                ),
        }
    )


reactivation_test_bootstrap_summary = pd.DataFrame(
    reactivation_test_bootstrap_summary
)


display(
    reactivation_test_bootstrap_summary
)

In [ ]:
# ================================================================
# 37.17 — DEVELOPMENT → TEST FEATURE DRIFT
# ================================================================

reactivation_test_drift_records = []


for feature in reactivation_numeric_features:

    development_values = pd.to_numeric(
        X_reactivation_development[
            feature
        ],
        errors="coerce"
    )


    test_values = pd.to_numeric(
        X_reactivation_test[
            feature
        ],
        errors="coerce"
    )


    development_mean = (
        development_values.mean()
    )


    test_mean = (
        test_values.mean()
    )


    pooled_std = np.sqrt(
        (
            development_values.std() ** 2
            +
            test_values.std() ** 2
        )
        / 2
    )


    if (
        pd.isna(pooled_std)
        or
        pooled_std == 0
    ):

        smd = np.nan

    else:

        smd = (
            test_mean
            -
            development_mean
        ) / pooled_std


    reactivation_test_drift_records.append(
        {
            "feature":
                feature,

            "development_mean":
                development_mean,

            "test_mean":
                test_mean,

            "SMD":
                smd,

            "absolute_SMD":
                abs(smd)
                if not pd.isna(smd)
                else np.nan,
        }
    )


reactivation_development_test_drift = (
    pd.DataFrame(
        reactivation_test_drift_records
    )
    .sort_values(
        "absolute_SMD",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    reactivation_development_test_drift
    .head(15)
)

In [ ]:
# ================================================================
# 37.18 — TEST DECISION PLACEHOLDER
# ================================================================

FINAL_REACTIVATION_DECISION = None

print(
    "Final reactivation deployment decision pending "
    "Test evidence review."
)

### Step 37 Summary — Final Untouched Temporal Test Evaluation

Step 37 performed the first and only evaluation of the frozen customer-reactivation model on the previously untouched future Test population.

The final model specification had already been frozen as a Full Random Forest before the Test population was opened.

The exact frozen specification was refitted using the combined Train and Validation development population and then evaluated once on the later Test episodes.

No model features, preprocessing rules, algorithms or hyperparameters were changed after Test results became available.

### Final Out-of-Time Performance

The frozen Random Forest achieved approximately:

- **PR-AUC: 0.615**
- **ROC-AUC: 0.699**
- **Brier score: 0.219**
- **Lift@10%: 1.56×**
- **Lift@20%: 1.56×**
- **Capture@10%: 15.65%**
- **Capture@20%: 31.30%**

The model therefore retained substantial predictive discrimination in genuinely future customer-inactivity episodes.

### Discrimination Remained Strong

Final Test ROC-AUC was approximately **0.699**.

Bootstrap analysis produced a 95% confidence interval of approximately **0.657 to 0.742**.

The entire interval remains well above random discrimination of 0.50.

The model therefore demonstrates robust out-of-time ranking signal.

### PR-AUC Must Be Interpreted Relative to Prevalence

Raw PR-AUC increased slightly from approximately **0.607 during Validation to 0.615 during Test**.

However, the underlying reactivation prevalence also increased from approximately **37.0% to 43.7%**.

PR-AUC relative to prevalence therefore declined from approximately **1.64× to 1.41×**, while normalised PR-AUC declined from approximately **0.376 to 0.316**.

The raw PR-AUC increase should therefore not be interpreted as improved discrimination.

Instead, the evidence indicates a moderate reduction in relative ranking strength while preserving substantial predictive value.

### Commercial Lift Survived Future Testing

Top-decile lift declined from approximately **2.00× during Validation to 1.56× on Test**.

Top-20% lift declined from approximately **1.81× to 1.56×**.

Despite this degradation, commercial ranking remained strong.

Bootstrap 95% confidence intervals were approximately:

- **Lift@10%: 1.30×–1.84×**
- **Lift@20%: 1.38×–1.75×**

Both intervals remain entirely above random lift of 1.0.

The model therefore demonstrates statistically credible future targeting value.

### Strong Score-Segment Separation

The highest-ranked Test deciles exhibited observed 90-day reactivation rates of approximately **68%**, compared with an overall Test reactivation rate of approximately **43.7%**.

The lowest-ranked decile reactivated at only approximately **13.3%**.

The model therefore creates substantial separation between customers with high and low natural reactivation propensity.

This segmentation is commercially meaningful.

### Calibration Remained Stable

Final Test mean absolute calibration error was approximately **5.76 percentage points**.

This is very similar to the approximately **5.87 percentage-point error observed during Validation**.

Probability quality therefore remained comparatively stable even as the population changed through time.

### The Model Generalised Despite Substantial Population Drift

Development-to-Test feature drift was large for several accumulated customer-history variables.

Examples included:

- tenure at prediction — **absolute SMD 0.99**
- historical maximum gap — **0.81**
- prediction month — **0.80**
- prior inactivity count — **0.76**
- prior-inactivity indicator — **0.71**
- historical order count — approximately **0.53**

These changes reflect a later and more mature customer population.

Despite this substantial drift, the frozen model maintained approximately 0.70 ROC-AUC and 1.56× Lift@20%.

This provides meaningful evidence that the predictive relationship generalises beyond the original customer-history distribution.

### Final Predictive Conclusion

The final Random Forest model successfully passed its untouched temporal holdout.

The model demonstrates:

- robust future discrimination;
- meaningful high-propensity concentration;
- strong separation of low-propensity customers;
- statistically credible commercial lift;
- stable calibration;
- and resilience under substantial population drift.

The predictive model itself therefore supports progression beyond exploratory modelling into monitored operational use.

The next requirement is to distinguish natural reactivation propensity from incremental response to intervention.



## STEP 38 — FINAL DEPLOYMENT DECISION, MODEL CARD & NOTEBOOK CLOSURE

In [ ]:
# ================================================================
# 38.1 — FINAL REACTIVATION MODEL DECISION
# ================================================================

FINAL_REACTIVATION_DECISION = "GO"


print(
    f"FINAL REACTIVATION MODEL DECISION: "
    f"{FINAL_REACTIVATION_DECISION}"
)

## Final Deployment Decision — GO

The frozen customer-reactivation model receives a final decision of:

> **GO**

GO means that the model has demonstrated sufficiently strong and robust out-of-time predictive performance to justify use as a monitored operational prioritisation tool.

GO does **not** mean:

- every individual prediction will be correct;
- the model should autonomously determine customer treatment;
- high propensity proves a customer will return;
- low propensity proves a customer is permanently churned;
- or marketing intervention has been proven to cause reactivation.

The model predicts **natural reactivation propensity**.

Its appropriate initial use is therefore customer segmentation and prioritisation within a controlled operating framework.

In [ ]:
# ================================================================
# 38.3 — FINAL REACTIVATION MODEL CARD
# ================================================================

final_reactivation_model_card = pd.DataFrame(
    {
        "Field": [
            "Business problem",
            "Prediction unit",
            "Prediction point",
            "Target",
            "Prediction horizon",
            "Model",
            "Training design",
            "Final Test episodes",
            "Final Test prevalence",
            "Final PR-AUC",
            "Final ROC-AUC",
            "Final Brier score",
            "Final Lift@10%",
            "Final Lift@20%",
            "Final Capture@10%",
            "Final Capture@20%",
            "Calibration error",
            "Final decision",
            "Primary operational use",
            "Primary limitation",
        ],

        "Final Position": [
            "Prioritise inactive customers by probability of natural reactivation",

            "One eligible 90-day inactivity episode",

            "Exactly 90 days after the customer's most recent qualifying purchase",

            "Another qualifying purchase during the following 90 days",

            "90 days",

            "Full Random Forest",

            "Chronological Train → Validation → untouched future Test",

            len(
                test_reactivation_data
            ),

            y_reactivation_test.mean(),

            final_reactivation_test_results[
                "PR-AUC"
            ].iloc[0],

            final_reactivation_test_results[
                "ROC-AUC"
            ].iloc[0],

            final_reactivation_test_results[
                "Brier Score"
            ].iloc[0],

            final_reactivation_test_results[
                "Lift @ 10%"
            ].iloc[0],

            final_reactivation_test_results[
                "Lift @ 20%"
            ].iloc[0],

            final_reactivation_test_results[
                "Capture @ 10%"
            ].iloc[0],

            final_reactivation_test_results[
                "Capture @ 20%"
            ].iloc[0],

            final_test_calibration_error,

            FINAL_REACTIVATION_DECISION,

            (
                "Monitored prioritisation of inactive customers "
                "for differentiated retention strategies"
            ),

            (
                "Propensity predicts natural return probability, "
                "not causal responsiveness to intervention"
            ),
        ],
    }
)


display(
    final_reactivation_model_card
)

### Recommended Operational Segments

Initial deployment should use broad propensity groups rather than highly granular score thresholds.

A practical framework is:

**High Reactivation Propensity**

Customers likely to return naturally.

Possible operational implication:

- low-cost reminders;
- standard communication;
- avoid unnecessary expensive incentives.

**Medium Reactivation Propensity**

Customers for whom future behaviour is less certain.

Possible operational implication:

- standardised retention treatment;
- experimentation with messaging and product recommendations.

**Low Reactivation Propensity**

Customers with a comparatively low natural probability of returning.

Possible operational implication:

- candidate population for controlled win-back intervention;
- stronger incentive testing;
- personalised retention treatment;
- or selective disengagement where intervention economics are unattractive.

These segments should initially be used to support decisions rather than automatically determine treatment.

In [ ]:
# ================================================================
# 38.5 — FINAL REACTIVATION MODEL LIMITATIONS
# ================================================================

final_reactivation_limitations = pd.DataFrame(
    [
        {
            "Limitation":
                "Propensity is not causal uplift",

            "Implication":
                "The model cannot identify who will change behaviour because of intervention."
        },

        {
            "Limitation":
                "Repeated customer episodes",

            "Implication":
                "The same customer may contribute several prediction opportunities."
        },

        {
            "Limitation":
                "Temporal customer maturity",

            "Implication":
                "Customer-history distributions change substantially through time."
        },

        {
            "Limitation":
                "Calendar dependence",

            "Implication":
                "Prediction month contains meaningful signal that may evolve in future periods."
        },

        {
            "Limitation":
                "Identified customers only",

            "Implication":
                "Customer-level predictions exclude transactions without usable customer identity."
        },

        {
            "Limitation":
                "Historical transaction features only",

            "Implication":
                "CRM engagement, promotion exposure and other behavioural variables are unavailable."
        },

        {
            "Limitation":
                "No intervention-cost optimisation",

            "Implication":
                "Predictive performance alone does not establish campaign profitability."
        },

        {
            "Limitation":
                "Historical retail environment",

            "Implication":
                "Future business conditions may differ from the observed period."
        },
    ]
)


display(
    final_reactivation_limitations
)

In [ ]:
# ================================================================
# 38.6 — PRODUCTION MONITORING REQUIREMENTS
# ================================================================

reactivation_monitoring_framework = pd.DataFrame(
    [
        {
            "Area":
                "Outcome prevalence",

            "Metric":
                "90-day reactivation rate"
        },

        {
            "Area":
                "Discrimination",

            "Metric":
                "ROC-AUC and PR-AUC"
        },

        {
            "Area":
                "Commercial ranking",

            "Metric":
                "Lift@10%, Lift@20%, segment reactivation rates"
        },

        {
            "Area":
                "Calibration",

            "Metric":
                "Predicted vs observed reactivation"
        },

        {
            "Area":
                "Feature drift",

            "Metric":
                "SMD / PSI on major predictors"
        },

        {
            "Area":
                "Score drift",

            "Metric":
                "Score distribution through time"
        },

        {
            "Area":
                "Campaign effectiveness",

            "Metric":
                "Treatment vs Control incremental reactivation"
        },

        {
            "Area":
                "Economics",

            "Metric":
                "Incremental margin and ROI"
        },
    ]
)


display(
    reactivation_monitoring_framework
)

## Next Analytical Evolution

The reactivation model predicts:

> **Who is naturally likely to return?**

The next-generation retention model should eventually answer:

> **Whose behaviour can intervention change?**

Once sufficient randomised treatment and control data have been collected, the business can develop an **uplift model** or heterogeneous treatment-effect model.

That model would distinguish:

- customers likely to return regardless of treatment;
- customers unlikely to return regardless of treatment;
- customers whose behaviour can be positively changed;
- and customers for whom intervention may be unnecessary or uneconomic.

This would convert reactivation propensity into true retention-spend optimisation.

In [ ]:
# ================================================================
# 38.8 — FINAL NOTEBOOK GOVERNANCE
# ================================================================

REACTIVATION_TEST_SET_CONSUMED = True
NOTEBOOK_3_COMPLETE = True


assert REACTIVATION_MODEL_FROZEN is True
assert REACTIVATION_TEST_SET_OPENED is True
assert REACTIVATION_TEST_SET_CONSUMED is True
assert FINAL_REACTIVATION_DECISION == "GO"


print(
    "✓ Frozen reactivation model selected."
)

print(
    "✓ Untouched temporal Test completed."
)

print(
    "✓ Final model decision: GO."
)

print(
    "✓ Test population permanently retired from development."
)

print(
    "✓ Notebook 3 complete."
)

# Final Notebook Conclusion

## Customer Reactivation Propensity

This notebook investigated whether customers who reach 90 days of inactivity can be meaningfully ranked according to their probability of returning during the following 90 days.

The analysis deliberately avoided treating inactivity as permanent churn.

Instead, each 90-day inactivity event was treated as a prediction opportunity.

After removing left-truncated customer histories, incomplete future-observation periods and temporally unsafe features, a chronological modelling population was created.

### Strong Historical Signal

Relationship depth proved to be a major predictor of future reactivation.

Customers with deeper purchase histories were materially more likely to return after inactivity than customers with only one or two historical orders.

However, richer historical features improved prediction beyond order count alone.

### Model Development

Several modelling approaches were compared, including:

- prevalence baselines;
- historical order count;
- Logistic Regression;
- reduced Logistic Regression;
- Random Forest;
- Gradient Boosting;
- HistGradientBoosting;
- and a Low-Drift Random Forest sensitivity model.

The Full Random Forest ultimately produced the strongest overall validation profile and was frozen before the final Test population was opened.

### Final Out-of-Time Result

On the untouched future Test population, the frozen Random Forest achieved approximately:

- **PR-AUC: 0.615**
- **ROC-AUC: 0.699**
- **Lift@10%: 1.56×**
- **Lift@20%: 1.56×**
- **Capture@10%: 15.65%**
- **Capture@20%: 31.30%**

Bootstrap confidence intervals for both Lift@10% and Lift@20% remained entirely above 1.0.

The model therefore retained statistically credible future targeting value.

### Strong Customer Segmentation

The highest-ranked customer groups exhibited approximately 68% observed reactivation compared with approximately 44% across the overall Test population.

The lowest-ranked decile reactivated at only approximately 13%.

This produces commercially meaningful separation between high- and low-propensity inactive customers.

### Robustness Under Population Change

The future Test population was substantially more mature than the development population.

Large shifts occurred in tenure, historical order count, inactivity history and purchase-gap behaviour.

Despite this drift, the frozen model retained approximately 0.70 ROC-AUC and 1.56× top-20% lift.

This provides strong evidence of genuine temporal generalisation.

## Final Decision

> **GO**

The model is suitable for monitored operational use as a reactivation-prioritisation layer.

It should not be interpreted as a causal treatment-effect model.

Its immediate role is to improve prioritisation of inactive customers while controlled experiments determine which interventions actually create incremental customer return and economic value.

The next generation of retention analytics should move from **reactivation propensity** toward **reactivation uplift**.

In [ ]:
# ================================================================
# EXPORT AUDIT — REACTIVATION NOTEBOOK
# ================================================================

reactivation_export_candidates = []


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    columns = list(
        obj.columns
    )


    columns_lower = [
        str(column).lower()
        for column in columns
    ]


    has_customer = (
        "customer"
        in columns_lower
    )


    reactivation_like_columns = [
        column
        for column in columns
        if (
            "reactiv"
            in str(column).lower()
            or
            (
                "prob"
                in str(column).lower()
            )
            or
            (
                "score"
                in str(column).lower()
            )
        )
    ]


    if (
        has_customer
        and
        reactivation_like_columns
    ):

        reactivation_export_candidates.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Columns":
                    columns,

                "Reactivation / Score Columns":
                    reactivation_like_columns,
            }
        )


reactivation_export_candidates = pd.DataFrame(
    reactivation_export_candidates
)


display(
    reactivation_export_candidates
)

In [ ]:
# ================================================================
# EXPORT AUDIT 2 — REACTIVATION CANDIDATE STRUCTURES
# ================================================================

reactivation_tables_to_inspect = [
    "reactivation_model_dataset",
    "train_reactivation_data",
    "validation_reactivation_data",
    "test_reactivation_data",
    "reactivation_development_data",
]


for name in reactivation_tables_to_inspect:

    if name not in globals():
        continue

    obj = globals()[name]

    print(
        "\n"
        +
        "=" * 80
    )

    print(
        f"{name}: {obj.shape}"
    )

    print(
        "=" * 80
    )

    for column in obj.columns:
        print(
            " -",
            column
        )

In [ ]:
# ================================================================
# EXPORT AUDIT 3 — FIND FINAL REACTIVATION SCORE OUTPUTS
# ================================================================

reactivation_scored_candidates = []


score_terms = [
    "probability",
    "prob",
    "propensity",
    "prediction",
    "predicted",
    "score",
]


for name, obj in list(globals().items()):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    matching_columns = [
        column
        for column in obj.columns
        if any(
            term
            in str(column).lower()
            for term in score_terms
        )
    ]


    # Require either customer or prediction_date context
    context_columns = [
        column
        for column in obj.columns
        if str(column).lower()
        in [
            "customer",
            "prediction_date",
            "snapshot_date",
            "score_date",
        ]
    ]


    if (
        matching_columns
        and
        context_columns
    ):

        reactivation_scored_candidates.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Context Columns":
                    context_columns,

                "Score-Like Columns":
                    matching_columns,

                "All Columns":
                    list(
                        obj.columns
                    ),
            }
        )


reactivation_scored_candidates = pd.DataFrame(
    reactivation_scored_candidates
)


display(
    reactivation_scored_candidates
)

In [ ]:
# ================================================================
# EXPORT AUDIT 4 — FIND REACTIVATION MODEL / PREDICTION OBJECTS
# SAFE VERSION
# ================================================================

reactivation_objects = []

search_terms = [
    "reactiv",
    "model",
    "forest",
    "rf",
    "pred",
    "prob",
    "score",
]


for name, obj in list(globals().items()):

    name_lower = str(name).lower()

    if not any(
        term in name_lower
        for term in search_terms
    ):
        continue

    # Skip DataFrames already inspected
    if isinstance(obj, pd.DataFrame):
        continue

    # Skip classes/types
    if isinstance(obj, type):
        continue

    # Safe length retrieval
    try:
        object_length = len(obj)
    except Exception:
        object_length = np.nan

    # Safe shape retrieval
    try:
        object_shape = getattr(
            obj,
            "shape",
            None
        )
    except Exception:
        object_shape = None

    reactivation_objects.append(
        {
            "Object": name,
            "Type": type(obj).__name__,
            "Length": object_length,
            "Shape": object_shape,
        }
    )


reactivation_objects = pd.DataFrame(
    reactivation_objects
)


if reactivation_objects.empty:

    print(
        "No matching non-DataFrame reactivation/model/prediction objects found."
    )

else:

    reactivation_objects = (
        reactivation_objects
        .sort_values(
            "Object"
        )
        .reset_index(
            drop=True
        )
    )

    display(
        reactivation_objects
    )

In [ ]:
# ================================================================
# EXPORT AUDIT 5 — FIND PREDICTION-SIZED ARRAYS
# ================================================================

known_reactivation_sizes = {
    599: "Test population",
    802: "Validation population",
    4052: "Train population",
    4854: "Development population",
    5453: "Full model dataset",
}


prediction_sized_objects = []


for name, obj in list(globals().items()):

    if isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    try:

        object_length = len(obj)

    except Exception:

        continue


    if object_length not in known_reactivation_sizes:
        continue


    # Only keep vector/array/series-like objects
    if isinstance(
        obj,
        (
            np.ndarray,
            pd.Series,
            list,
            tuple,
        )
    ):

        arr = np.asarray(obj)

        prediction_sized_objects.append(
            {
                "Object":
                    name,

                "Type":
                    type(obj).__name__,

                "Length":
                    object_length,

                "Likely Population":
                    known_reactivation_sizes[
                        object_length
                    ],

                "Shape":
                    arr.shape,

                "Min":
                    (
                        np.nanmin(arr)
                        if (
                            np.issubdtype(
                                arr.dtype,
                                np.number
                            )
                            and
                            arr.size > 0
                        )
                        else np.nan
                    ),

                "Max":
                    (
                        np.nanmax(arr)
                        if (
                            np.issubdtype(
                                arr.dtype,
                                np.number
                            )
                            and
                            arr.size > 0
                        )
                        else np.nan
                    ),
            }
        )


prediction_sized_objects = pd.DataFrame(
    prediction_sized_objects
)


if prediction_sized_objects.empty:

    print(
        "No prediction-sized arrays found."
    )

else:

    display(
        prediction_sized_objects
    )

In [ ]:
# ================================================================
# EXPORT AUDIT 6 — FIND FITTED SCIKIT-LEARN ESTIMATORS
# ================================================================

fitted_estimators = []


for name, obj in list(globals().items()):

    if isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    has_predict = hasattr(
        obj,
        "predict"
    )

    has_predict_proba = hasattr(
        obj,
        "predict_proba"
    )

    has_fit = hasattr(
        obj,
        "fit"
    )


    if (
        has_predict
        or
        has_predict_proba
    ):

        fitted_estimators.append(
            {
                "Object":
                    name,

                "Type":
                    type(obj).__name__,

                "Has Predict":
                    has_predict,

                "Has Predict Proba":
                    has_predict_proba,

                "Has Fit":
                    has_fit,

                "Feature Count":
                    getattr(
                        obj,
                        "n_features_in_",
                        np.nan
                    ),
            }
        )


fitted_estimators = pd.DataFrame(
    fitted_estimators
)


if fitted_estimators.empty:

    print(
        "No fitted estimator-like objects found."
    )

else:

    display(
        fitted_estimators
    )

In [ ]:
# ================================================================
# EXPORT — REACTIVATION PROPENSITY MODEL OUTPUT
# ================================================================

from pathlib import Path

MODEL_OUTPUT_DIR = Path("model_outputs")

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------
# Safety: Test rows and final probabilities
# must align exactly
# ------------------------------------------------

assert (
    len(test_reactivation_data)
    ==
    len(reactivation_test_probabilities)
), (
    "Test dataset and Reactivation probability "
    "vector have different lengths."
)


# ------------------------------------------------
# Construct clean historical scoring output
# ------------------------------------------------

reactivation_scores = (
    test_reactivation_data[
        [
            "customer",
            "invoice",
            "prediction_date",
            "prediction_period",
            "inactivity_episode_number",
            "historical_order_count",
            "historical_revenue",
            "historical_units",
            "historical_average_order_value",
            "tenure_at_prediction_days",
            "historical_average_gap_days",
            "prior_inactivity_episode_count",
            "prior_reactivation_count",
            "prior_reactivation_rate",
            "last_order_value",
            "last_order_region",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------
# Attach FINAL held-out Test probabilities
# ------------------------------------------------

reactivation_scores[
    "reactivation_probability"
] = np.asarray(
    reactivation_test_probabilities,
    dtype=float
)


# ------------------------------------------------
# Standardise customer identifier
# ------------------------------------------------

reactivation_scores[
    "customer"
] = (
    pd.to_numeric(
        reactivation_scores[
            "customer"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


reactivation_scores[
    "prediction_date"
] = pd.to_datetime(
    reactivation_scores[
        "prediction_date"
    ]
)


# ------------------------------------------------
# Explicit provenance
# ------------------------------------------------

reactivation_scores[
    "reactivation_model_name"
] = str(
    FINAL_REACTIVATION_MODEL_NAME
)


reactivation_scores[
    "reactivation_model_scope"
] = (
    "Probability of reactivation within 90 days "
    "after eligible inactivity episode"
)


reactivation_scores[
    "reactivation_score_provenance"
] = (
    "historical held-out Test episode scoring"
)


# ------------------------------------------------
# Safety checks
# ------------------------------------------------

assert (
    reactivation_scores[
        "customer"
    ]
    .notna()
    .all()
)


assert (
    reactivation_scores[
        "prediction_date"
    ]
    .notna()
    .all()
)


assert (
    reactivation_scores[
        "reactivation_probability"
    ]
    .between(
        0,
        1
    )
    .all()
)


assert (
    reactivation_scores[
        [
            "customer",
            "prediction_date",
            "inactivity_episode_number",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
), (
    "Duplicate customer / prediction-date / episode rows found."
)


# ------------------------------------------------
# Save persistent file
# ------------------------------------------------

reactivation_output_path = (
    MODEL_OUTPUT_DIR
    /
    "reactivation_scores.parquet"
)


reactivation_scores.to_parquet(
    reactivation_output_path,
    index=False
)


# ------------------------------------------------
# Export audit
# ------------------------------------------------

print(
    "✓ Reactivation scores exported."
)

print(
    reactivation_output_path.resolve()
)

print(
    f"Rows: {len(reactivation_scores):,}"
)

print(
    f"Customers: "
    f"{reactivation_scores['customer'].nunique():,}"
)

print(
    f"Prediction date range: "
    f"{reactivation_scores['prediction_date'].min()} "
    f"to "
    f"{reactivation_scores['prediction_date'].max()}"
)

print(
    f"Probability range: "
    f"{reactivation_scores['reactivation_probability'].min():.4f} "
    f"to "
    f"{reactivation_scores['reactivation_probability'].max():.4f}"
)


display(
    reactivation_scores.head(
        10
    )
)